In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load files (they are in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Helper: add a localized datetime column (UTC->UTC+8) for internal use (not required in samples)
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        # ensure numeric
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        df['ts_utc8'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# METRIC inventory
metric_count = metric_df.shape[0]
metric_cmdb_counts = metric_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
metric_kpi_counts = metric_df['kpi_name'].value_counts().rename_axis('kpi_name').reset_index(name='count').head(50)
metric_sample = metric_df[['timestamp', 'cmdb_id', 'kpi_name', 'value']].head(5)

# TRACE inventory
trace_count = trace_df.shape[0]
trace_cmdb_counts = trace_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
trace_name_counts = trace_df['trace_name'].value_counts().rename_axis('trace_name').reset_index(name='count').head(50)
trace_sample = trace_df[['timestamp', 'cmdb_id', 'trace_name', 'value']].head(5)

# LOG inventory
log_count = log_df.shape[0]
log_cmdb_counts = log_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
log_name_counts = log_df['log_name'].value_counts().rename_axis('log_name').reset_index(name='count').head(50)
log_sample = log_df[['timestamp', 'cmdb_id', 'log_name', 'value']].head(5)

# ERROR_LOGS inventory
error_count = error_df.shape[0]
error_cmdb_counts = error_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
error_top_messages = error_df['message'].value_counts().rename_axis('message').reset_index(name='count').head(10)
error_sample = error_df[['timestamp', 'cmdb_id', 'message']].head(5)

# Compact summaries (returned variables)
metric_count, metric_cmdb_counts, metric_kpi_counts, metric_sample, \
trace_count, trace_cmdb_counts, trace_name_counts, trace_sample, \
log_count, log_cmdb_counts, log_name_counts, log_sample, \
error_count, error_cmdb_counts, error_top_messages, error_sample

```
Out[1]:
```
```python
# Build a concise plain-English summary using previously loaded inventory variables.
# Assumes variables from prior execution exist: metric_count, metric_cmdb_counts, metric_kpi_counts, 
# trace_count, trace_cmdb_counts, trace_name_counts, log_count, log_cmdb_counts, log_name_counts,
# error_count, error_cmdb_counts, error_top_messages

def fmt_top(df, key_col, cnt_col='count', n=6):
    if df is None or df.shape[0]==0:
        return "none"
    rows = df.head(n).itertuples(index=False)
    return ", ".join(f"{getattr(r, key_col)}({getattr(r, cnt_col)})" for r in rows)

summary = (
    "Inventory summary (compact):\n\n"
    f"1) metric.csv: total rows = {metric_count}\n"
    f"   Top CMDBs: {fmt_top(metric_cmdb_counts, 'cmdb_id', 'count', 6)}\n"
    f"   Top KPIs (sample top 5): {fmt_top(metric_kpi_counts, 'kpi_name', 'count', 5)}\n"
    "   Sample rows: first 5 shown previously.\n\n"
    f"2) trace.csv: total rows = {trace_count}\n"
    f"   Top CMDBs: {fmt_top(trace_cmdb_counts, 'cmdb_id', 'count', 6)}\n"
    f"   Top trace names: {fmt_top(trace_name_counts, 'trace_name', 'count', 5)}\n"
    "   Sample rows: first 5 shown previously.\n\n"
    f"3) log.csv: total rows = {log_count}\n"
    f"   Top CMDBs: {fmt_top(log_cmdb_counts, 'cmdb_id', 'count', 8)}\n"
    f"   Top log names: {fmt_top(log_name_counts, 'log_name', 'count', 5)}\n"
    "   Sample rows: first 5 shown previously.\n\n"
    f"4) error_logs.csv: total rows = {error_count}\n"
    f"   CMDB distribution: {fmt_top(error_cmdb_counts, 'cmdb_id', 'count', 6)}\n"
    f"   Top messages: {fmt_top(error_top_messages, 'message', 'count', 10)}\n\n"
    "Notes: error_logs.csv appears empty (no messages). All samples and top lists were kept compact."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(27812,          cmdb_id  count
0        Mysql02   4019
1        Mysql01   3966
2        Redis01   2244
3        Redis02   2205
4       Tomcat02   1619
..           ...    ...
24  ServiceTest7    124
25      dockerA2     65
26      dockerB2     30
27      dockerA1     29
28      dockerB1     18

[29 rows x 2 columns],                                              kpi_name  count
0                  OSLinux-OSLinux_ZABBIX_Host_Uptime    368
1       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    368
2          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    368
3      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    367
4        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    365
..                                                ...    ...
45  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...    176
46    OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent    176
47        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity    176
48  OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPe...    176
49  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    175

[50 rows x 2 columns],     timestamp cmdb_id                                     kpi_name         value
0  1615384800    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1  1615384800    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.140130e+01
2  1615384800    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  7.284311e+08
3  1615384800    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.792518e+07
4  1615384800    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad  2.145000e-01, 6870,      cmdb_id  count
0       MG01   1173
1       MG02   1170
2   Tomcat03    456
3       IG02    453
4       IG01    453
5   Tomcat01    453
6   Tomcat02    453
7   Tomcat04    453
8   dockerA2    453
9   dockerB2    453
10  dockerA1    450
11  dockerB1    450,                            trace_name  count
0            trace.self.duration_mean    360
1             trace.self.duration_p95    360
2                trace.self.row_count    360
3             trace.to_MG01.row_count    240
4         trace.to_MG01.duration_mean    240
..                                ...    ...
45      trace.from_dockerB2.row_count     60
46   trace.from_dockerB2.duration_p95     60
47  trace.from_dockerB2.duration_mean     60
48      trace.from_dockerA2.row_count     60
49  trace.from_dockerB1.duration_mean     60

[50 rows x 2 columns],     timestamp cmdb_id                       trace_name       value
0  1615384800    IG01         trace.self.duration_mean    0.095682
1  1615384800    IG01          trace.self.duration_p95    0.565000
2  1615384800    IG01             trace.self.row_count  576.000000
3  1615384800    IG01  trace.to_Tomcat01.duration_mean    0.155014
4  1615384800    IG01   trace.to_Tomcat01.duration_p95    1.023000, 438,     cmdb_id  count
0  Tomcat02     73
1  Tomcat01     70
2  Tomcat03     70
3  Tomcat04     69
4  apache01     62
5  apache02     62
6   Mysql01     32,            log_name  count
0   log.error_count    201
1     log.row_count    201
2  log.gc_pause_max     36,     timestamp   cmdb_id          log_name   value
0  1615384800   Mysql01   log.error_count    0.00
1  1615384800   Mysql01     log.row_count    3.00
2  1615384800  Tomcat01   log.error_count    0.00
3  1615384800  Tomcat01  log.gc_pause_max    0.08
4  1615384800  Tomcat01     log.row_count  152.00, 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [message, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df from kernel state
tz = pytz.timezone('Asia/Shanghai')

# Ensure ts_utc8 exists (create if missing)
if 'ts_utc8' not in metric_df.columns:
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['ts_utc8'] = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# Components of interest
components = [
    'apache01','apache02',
    'Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02',
    'IG01','IG02',
    'Mysql01','Mysql02',
    'Redis01','Redis02'
]

# Filter metric_df to these components
filtered_metric = metric_df[metric_df['cmdb_id'].isin(components)].copy()

# Global group statistics (over entire available series) per (cmdb_id, kpi_name)
group_stats = (
    filtered_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        first_ts=('ts_utc8', 'min'),
        last_ts=('ts_utc8', 'max'),
        mean=('value', 'mean'),
        max=('value', 'max'),
        p95=('value', lambda s: float(np.nanpercentile(s.dropna(), 95)) if s.dropna().size>0 else np.nan),
        p5=('value', lambda s: float(np.nanpercentile(s.dropna(), 5)) if s.dropna().size>0 else np.nan)
    )
)

# Incident window in UTC+8
win_start = pd.Timestamp('2021-03-10 22:00:00').tz_localize(tz)
win_end   = pd.Timestamp('2021-03-10 22:30:00').tz_localize(tz)

# Rows inside the incident window
in_window = filtered_metric[(filtered_metric['ts_utc8'] >= win_start) & (filtered_metric['ts_utc8'] <= win_end)].copy()

# in-window stats per group
in_window_stats = (
    in_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        in_window_count=('value','count'),
        in_window_first_ts=('ts_utc8','min')
    )
)

# Merge global stats with in-window stats (fill zeros/NaT for groups with no in-window points)
merged = pd.merge(group_stats, in_window_stats, on=['cmdb_id','kpi_name'], how='left')
merged['in_window_count'] = merged['in_window_count'].fillna(0).astype(int)
# in_window_first_ts keep as NaT if no points

# Prepare display columns and formatting: keep timestamps as ISO strings for compactness
display = merged.copy()
display['first_ts'] = display['first_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
display['last_ts']  = display['last_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
display['in_window_first_ts'] = display['in_window_first_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
# If NaT -> result of strftime is NaN, replace with None for clarity
display['in_window_first_ts'] = display['in_window_first_ts'].where(pd.notnull(display['in_window_first_ts']), None)

# 1) Top 50 groups sorted by P95 descending
results_top_p95 = display.sort_values('p95', ascending=False).head(50)[
    ['cmdb_id','kpi_name','count','first_ts','last_ts','mean','max','p95','p5','in_window_count','in_window_first_ts']
].reset_index(drop=True)

# 2) Groups with in_window_count > 0, sorted by in_window_count desc (limit 50)
results_in_window = display[display['in_window_count']>0].sort_values('in_window_count', ascending=False).head(50)[
    ['cmdb_id','kpi_name','count','first_ts','last_ts','mean','max','p95','p5','in_window_count','in_window_first_ts']
].reset_index(drop=True)

# Keep full intermediate tables for later steps: group_stats, in_window_stats, merged are stored
group_stats.shape, in_window_stats.shape, merged.shape, results_top_p95, results_in_window

```
Out[2]:
```
Summary (plain English):

- Scope & window: I analyzed metric.csv for the 14 components you listed, computing per-group (cmdb_id, kpi_name) global stats and in-window counts for the incident window 2021-03-10 22:00:00 — 2021-03-10 22:30:00 (Asia/Shanghai, UTC+8).

- Overall coverage:
  - Number of groups (cmdb_id, kpi_name) processed: 1,629.
  - Every one of those 1,629 groups had at least one data point inside the incident window (in_window_count > 0).
  - Many groups have full coverage across the window (in_window_count = 30 or 31), and the earliest in-window timestamps are typically 2021-03-10 22:00:00 or 2021-03-10 22:01:00 (UTC+8).

- Top groups by P95 (high-level highlights from the Top-50 P95 table):
  - Several Redis and OS network KPIs dominate the highest P95 values. Examples:
    - Redis01 OSLinux-..._NETPacketsIn — p95 ≈ 5.8945e+09, in_window_count = 16, in_window_first_ts = 2021-03-10 22:00:00.
    - Redis01 OSLinux-..._NETPacketsOut — p95 ≈ 5.3544e+09, in_window_count = 16, in_window_first_ts = 2021-03-10 22:00:00.
    - Redis01 redis total_commands_processed — p95 ≈ 5.2154e+09, in_window_count = 30, in_window_first_ts = 2021-03-10 22:01:00.
    - Several redis maxmemory/use_memory metrics show p95 values near 4.294967e+09 or ~5.04e+07 (used memory peak).
  - The Top-50 list contains a mix of OS-level, JVM, Redis and other metrics with varying magnitudes; p95/p5 and mean/max are included in the table.

- Groups with data inside the window:
  - The filtered table of groups with in-window data (ordered by in_window_count) shows many groups with in_window_count = 31 (full-minute coverage for the 31-minute inclusive window) and many with 30 points (starting at 22:01).
  - Example rows from that filtered list include apache02, IG01, IG02, Tomcat04, etc., typically with in_window_first_ts = 2021-03-10 22:00:00 and in_window_count = 31.

- Stored variables for follow-up:
  - Full intermediate results are available in the kernel as: group_stats, in_window_stats, merged.
  - Compact result tables are available as: results_top_p95 (Top 50 by P95) and results_in_window (groups with in-window data, sorted by in_window_count).

If you want, I can:
- Display the Top-50 P95 table or the in-window filtered table (head or specific rows).
- Show counts distribution of in_window_count (how many groups have 31 vs 30 vs fewer points).

The original code execution output of IPython Kernel is also provided below for reference:

((1629, 9), (1629, 4), (1629, 11),     cmdb_id                                           kpi_name  count                 first_ts                  last_ts          mean           max           p95            p5  in_window_count       in_window_first_ts
0   Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     16  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  2.578505e+09  5.894721e+09  5.894520e+09  9.310000e+02               16  2021-03-10 22:00:00 CST
1   Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut     16  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  2.342220e+09  5.354538e+09  5.354358e+09  8.267500e+02               16  2021-03-10 22:00:00 CST
2   Redis01  redis-Redis_6379_Redis  (total_commands_proces...     30  2021-03-10 22:01:00 CST  2021-03-10 22:30:00 CST  5.214674e+09  5.215492e+09  5.215403e+09  5.213906e+09               30  2021-03-10 22:01:00 CST
3   Redis01                redis-Redis_6379_Redis  (maxmemory)     30  2021-03-10 22:01:00 CST  2021-03-10 22:30:00 CST  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09               30  2021-03-10 22:01:00 CST
4   Redis02                redis-Redis_6379_Redis  (maxmemory)     30  2021-03-10 22:01:00 CST  2021-03-10 22:30:00 CST  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09               30  2021-03-10 22:01:00 CST
..      ...                                                ...    ...                      ...                      ...           ...           ...           ...           ...              ...                      ...
45  Redis02  redis-Redis_6379_Redis  (total_connections_rec...     30  2021-03-10 22:01:00 CST  2021-03-10 22:30:00 CST  7.331625e+07  7.332015e+07  7.331974e+07  7.331274e+07               30  2021-03-10 22:01:00 CST
46     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  5.792518e+07  5.792518e+07  5.792518e+07  5.792518e+07               31  2021-03-10 22:00:00 CST
47     IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  5.769137e+07  5.771062e+07  5.771062e+07  5.768714e+07               31  2021-03-10 22:00:00 CST
48  Redis01         redis-Redis_6379_Redis  (used_memory_peak)     30  2021-03-10 22:01:00 CST  2021-03-10 22:30:00 CST  5.041321e+07  5.041321e+07  5.041321e+07  5.041321e+07               30  2021-03-10 22:01:00 CST
49  Redis01  redis-Redis_6379_Redis  (total_connections_rec...     30  2021-03-10 22:01:00 CST  2021-03-10 22:30:00 CST  3.277023e+07  3.277594e+07  3.277549e+07  3.276497e+07               30  2021-03-10 22:01:00 CST

[50 rows x 11 columns],      cmdb_id                                           kpi_name  count                 first_ts                  last_ts          mean           max           p95            p5  in_window_count       in_window_first_ts
0   apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31  2021-03-10 22:00:00 CST
1   apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  6.584350e+06  6.585253e+06  6.585162e+06  6.583538e+06               31  2021-03-10 22:00:00 CST
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09               31  2021-03-10 22:00:00 CST
3   apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31  2021-03-10 22:00:00 CST
4       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  4.404401e+08  8.543139e+08  8.286598e+08  7.040201e+07               31  2021-03-10 22:00:00 CST
..       ...                                                ...    ...                      ...                      ...           ...           ...           ...           ...              ...                      ...
45  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31  2021-03-10 22:00:00 CST
46  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31  2021-03-10 22:00:00 CST
47  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31  2021-03-10 22:00:00 CST
48      IG02           JVM-Threads_7778_JVM_ThreadCount_Threads     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  7.109677e+01  7.600000e+01  7.600000e+01  6.700000e+01               31  2021-03-10 22:00:00 CST
49      IG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST  6.584704e+06  6.585607e+06  6.585517e+06  6.583893e+06               31  2021-03-10 22:00:00 CST

[50 rows x 11 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# reuse variables from kernel: metric_df, merged (has p95 and p5 for groups), filtered_metric if available
tz = pytz.timezone('Asia/Shanghai')

# Components list (reuse from earlier)
components = [
    'apache01','apache02',
    'Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02',
    'IG01','IG02',
    'Mysql01','Mysql02',
    'Redis01','Redis02'
]

# Ensure timestamp columns exist and are timezone-aware UTC+8
if 'ts_utc8' not in metric_df.columns:
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['ts_utc8'] = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive) in UTC+8
win_start = pd.Timestamp('2021-03-10 22:00:00').tz_localize(tz)
win_end   = pd.Timestamp('2021-03-10 22:30:00').tz_localize(tz)

# 1) Filter metric.csv to the incident window and 14 components
in_window = metric_df[
    (metric_df['cmdb_id'].isin(components)) &
    (metric_df['ts_utc8'] >= win_start) &
    (metric_df['ts_utc8'] <= win_end)
].copy()

# 2) Join each in-window row with group's global p95 and p5 (use merged or group_stats)
# merged from prior step contains p95,p5 per (cmdb_id,kpi_name)
if 'merged' not in globals():
    raise RuntimeError("Expected group-level thresholds in variable 'merged' from prior step.")

thresholds = merged[['cmdb_id','kpi_name','p95','p5']].copy()
in_window_marked = pd.merge(in_window, thresholds, on=['cmdb_id','kpi_name'], how='left')

# 3) Mark anomalies
# high-anomaly if value >= p95; low-anomaly if value <= p5
# handle NaNs: comparison yields False
in_window_marked['high_anom'] = in_window_marked['value'] >= in_window_marked['p95']
in_window_marked['low_anom']  = in_window_marked['value'] <= in_window_marked['p5']

# 4) Compute deviations:
# high_dev = (value - p95)/p95 for high anomalies, unless p95==0 then use abs(value - p95)
# low_dev  = (p5 - value)/p5 for low anomalies, unless p5==0 then use abs(p5 - value)
p95 = in_window_marked['p95'].fillna(0.0)
p5  = in_window_marked['p5'].fillna(0.0)
val = in_window_marked['value'].astype(float)

# compute high_dev and low_dev, only valid for rows flagged as such, else NaN
in_window_marked['high_dev'] = np.where(
    in_window_marked['high_anom'],
    np.where(p95 != 0, (val - p95) / p95, np.abs(val - p95)),
    np.nan
)

in_window_marked['low_dev'] = np.where(
    in_window_marked['low_anom'],
    np.where(p5 != 0, (p5 - val) / p5, np.abs(p5 - val)),
    np.nan
)

# 5) Aggregate per group
agg = in_window_marked.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    in_window_count=('value','count'),
    anomaly_count=('high_anom', lambda s: int(s.sum()) + int((in_window_marked.loc[s.index,'low_anom']).sum()) - int(((in_window_marked.loc[s.index,'high_anom']) & (in_window_marked.loc[s.index,'low_anom'])).sum())),
    # The above counts rows marked high OR low. We compute as sum(high) + sum(low) - sum(both)
    earliest_anom_ts=('ts_utc8', lambda s: s.loc[(in_window_marked.loc[s.index,'high_anom'] | in_window_marked.loc[s.index,'low_anom'])].min() if ((in_window_marked.loc[s.index,'high_anom'] | in_window_marked.loc[s.index,'low_anom']).any()) else pd.NaT),
    max_value_in_window=('value','max'),
    min_value_in_window=('value','min'),
    max_high_deviation=('high_dev', 'max'),
    max_low_deviation=('low_dev', 'max')
)

# Post-process earliest_anom_ts to string format and keep None for NaT
agg['earliest_anom_ts'] = agg['earliest_anom_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
agg['earliest_anom_ts'] = agg['earliest_anom_ts'].where(pd.notnull(agg['earliest_anom_ts']), None)

# Replace possible NaN deviations with 0.0 for clearer comparison (but keep NaN possible if desired)
# We'll keep NaN to indicate no such anomaly type; for sorting by max deviation we need numeric fill 0
agg['max_high_deviation_numeric'] = agg['max_high_deviation'].fillna(0.0)
agg['max_low_deviation_numeric']  = agg['max_low_deviation'].fillna(0.0)
agg['max_abs_deviation'] = agg[['max_high_deviation_numeric','max_low_deviation_numeric']].max(axis=1)

# Filter groups with anomaly_count > 0
anom_groups = agg[agg['anomaly_count'] > 0].copy()

# Prepare output tables (compact, selected columns)
cols_out = [
    'cmdb_id','kpi_name','anomaly_count','earliest_anom_ts','in_window_count',
    'max_value_in_window','min_value_in_window','max_high_deviation','max_low_deviation'
]

# a) Top 20 by earliest_anom_ts ascending
results_earliest = anom_groups.sort_values(['earliest_anom_ts','cmdb_id'], na_position='last').head(20)[cols_out].reset_index(drop=True)

# b) Top 20 by max_abs_deviation descending
results_severity = anom_groups.sort_values('max_abs_deviation', ascending=False).head(20)[cols_out].reset_index(drop=True)

# Total number of groups with anomaly_count > 0
total_anom_groups = anom_groups.shape[0]

# Store full anomaly-marked in-window table for next steps
in_window_marked.shape, results_earliest, results_severity, total_anom_groups, in_window_marked

```
Out[3]:
```
```python
# Build a concise plain-English summary string using previously computed variables.
# Assumes variables from prior execution exist: results_earliest, results_severity, total_anom_groups, in_window_marked, merged

def top_examples(df, n=3):
    rows = []
    for r in df.head(n).itertuples(index=False):
        rows.append(f"{r.cmdb_id} | {r.kpi_name} | anomalies={int(r.anomaly_count)} | earliest={r.earliest_anom_ts} | in_window={int(r.in_window_count)}")
    return rows

summary_lines = []
summary_lines.append("Anomaly analysis summary (incident window 2021-03-10 22:00—22:30 UTC+8):")
summary_lines.append("")
summary_lines.append(f"- Total groups evaluated (cmdb_id,kpi_name) for the 14 components: {merged.shape[0]}")
summary_lines.append(f"- Total in-window rows analyzed: {in_window_marked.shape[0]}")
summary_lines.append(f"- Number of groups with anomaly_count > 0: {int(total_anom_groups)}")
summary_lines.append("")
summary_lines.append("Top groups by earliest anomaly (examples):")
summary_lines += ["  - " + ex for ex in top_examples(results_earliest, 5)]
summary_lines.append("")
summary_lines.append("Top groups by max deviation severity (examples):")
summary_lines += ["  - " + ex for ex in top_examples(results_severity, 5)]
summary_lines.append("")
summary_lines.append("Notes:")
summary_lines.append(" - 'high' if value >= group's global p95, 'low' if value <= group's global p5.")
summary_lines.append(" - Many groups show full coverage in the window (30–31 points).")
summary_lines.append(" - Examples include JVM memory metrics on IG01 with earliest anomalies at 2021-03-10 22:00,")
summary_lines.append("   and several MySQL metrics (Mysql01/Mysql02) showing the largest relative deviations.")
summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((26306, 11),    cmdb_id                                           kpi_name  anomaly_count         earliest_anom_ts  in_window_count  max_value_in_window  min_value_in_window  max_high_deviation  max_low_deviation
0     IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-10 22:00:00 CST               31         2.040136e+09         2.040136e+09            0.000000           0.000000
1     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31  2021-03-10 22:00:00 CST               31         5.792518e+07         5.792518e+07            0.000000           0.000000
2     IG01          JVM-Operating System_7778_JVM_JVM_CPULoad              4  2021-03-10 22:00:00 CST               21         2.168000e-01         1.654000e-01            0.010723           0.004214
3     IG01                    JVM-Runtime_7778_JVM_JVM_Uptime              4  2021-03-10 22:00:00 CST               21         4.656450e+05         4.638440e+05            0.000258           0.000259
4     IG01                            OSLinux-CPU_CPU_CPULoad             14  2021-03-10 22:00:00 CST               31         1.160000e+00         1.000000e+00            0.004329           0.000000
5     IG01            OSLinux-OSLinux_FILESYSTEM_-_FSCapacity             11  2021-03-10 22:00:00 CST               16         3.026600e+01         3.026590e+01            0.000000           0.000000
6     IG01    OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent             16  2021-03-10 22:00:00 CST               16         1.636640e+01         1.636637e+01            0.000000           0.000000
7     IG01        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity             16  2021-03-10 22:00:00 CST               16         1.541700e+01         1.541695e+01            0.000000           0.000000
8     IG01  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...             16  2021-03-10 22:00:00 CST               16         4.715000e-01         4.714970e-01            0.000000           0.000000
9     IG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity             16  2021-03-10 22:00:00 CST               16         4.268120e-01         4.268000e-01            0.000000           0.000000
10    IG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...             16  2021-03-10 22:00:00 CST               16         4.200000e-03         4.196000e-03            0.000000           0.000000
11    IG01  OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPe...             16  2021-03-10 22:00:00 CST               16         2.116390e-01         2.116000e-01            0.000000           0.000000
12    IG01         OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity             16  2021-03-10 22:00:00 CST               16         9.875200e-01         9.875000e-01            0.000000           0.000000
13    IG01  OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPer...             16  2021-03-10 22:00:00 CST               16         1.590000e-02         1.586900e-02            0.000000           0.000000
14    IG01      OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity              2  2021-03-10 22:00:00 CST               16         2.785930e+01         2.776900e+01            0.000164           0.000194
15    IG01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsed...              8  2021-03-10 22:00:00 CST               16         9.470000e-02         9.450000e-02            0.000000           0.000000
16    IG01  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...             11  2021-03-10 22:00:00 CST               18         3.290000e-01         3.279000e-01            0.000000           0.000000
17    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             16  2021-03-10 22:00:00 CST               16         1.000000e+00         0.000000e+00            0.000000           0.000000
18    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...             16  2021-03-10 22:00:00 CST               16         1.000000e-01         0.000000e+00            0.000000           0.000000
19    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             14  2021-03-10 22:00:00 CST               16         2.283300e+00         0.000000e+00            2.167620           0.000000,      cmdb_id                                         kpi_name  anomaly_count         earliest_anom_ts  in_window_count  max_value_in_window  min_value_in_window  max_high_deviation  max_low_deviation
0    Mysql02                Mysql-MySQL_3306_Innodb data read             20  2021-03-10 22:00:00 CST               20             273.0667                  0.0           19.000000                0.0
1    Mysql02               Mysql-MySQL_3306_Innodb data reads             20  2021-03-10 22:00:00 CST               20               0.0167                  0.0           19.000000                0.0
2   Tomcat02   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             20  2021-03-10 22:01:00 CST               20               1.0000                  0.0           19.000000                0.0
3    Mysql01            Mysql-MySQL_3306_Innodb Row Lock Time             20  2021-03-10 22:00:00 CST               20               0.0328                  0.0           19.000000                0.0
4    Mysql01           Mysql-MySQL_3306_Innodb Row Lock Waits             20  2021-03-10 22:00:00 CST               20               0.0164                  0.0           19.000000                0.0
5    Mysql02               Mysql-MySQL_3306_Innodb pages read             20  2021-03-10 22:00:00 CST               20               0.0167                  0.0           19.000000                0.0
6    Mysql02             Mysql-MySQL_3306_Max trx rows locked             20  2021-03-10 22:00:00 CST               20               7.0000                  0.0           19.000000                0.0
7    Mysql02        Mysql-MySQL_3306_Opened table definitions             20  2021-03-10 22:00:00 CST               20               0.1667                  0.0           19.000000                0.0
8    Mysql01                      Mysql-MySQL_3306_Com Insert             20  2021-03-10 22:00:00 CST               20               0.2623                  0.0           19.000000                0.0
9       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             20  2021-03-10 22:01:00 CST               20               1.0000                  0.0           19.000000                0.0
10   Mysql02                 Mysql-MySQL_3306_Aborted Clients             20  2021-03-10 22:00:00 CST               20               0.0333                  0.0           19.000000                0.0
11   Mysql01            Mysql-MySQL_3306_Innodb pages created             20  2021-03-10 22:00:00 CST               20               0.0164                  0.0           19.000000                0.0
12   Mysql02        Mysql-MySQL_3306_Innodb buffer pool reads             20  2021-03-10 22:00:00 CST               20               0.0167                  0.0           19.000000                0.0
13   Mysql02                      Mysql-MySQL_3306_Sort Range             20  2021-03-10 22:00:00 CST               20               0.1167                  0.0           19.000000                0.0
14   Mysql02               Mysql-MySQL_3306_Handler Read Next             20  2021-03-10 22:00:00 CST               20               0.1167                  0.0           19.000000                0.0
15   Mysql02                Mysql-MySQL_3306_Handler Read Rnd             20  2021-03-10 22:00:00 CST               20               0.1167                  0.0           19.000000                0.0
16   Mysql02                   Mysql-MySQL_3306_Opened Tables             18  2021-03-10 22:00:00 CST               20               0.6833                  0.0            9.384498                0.0
17   Mysql02         Mysql-MySQL_3306_Table open cache misses             18  2021-03-10 22:00:00 CST               20               0.6833                  0.0            9.384498                0.0
18   Redis02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             16  2021-03-10 22:00:00 CST               16               6.0000                  0.0            3.000000                0.0
19   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps             16  2021-03-10 22:00:00 CST               16               1.2333                  0.0            3.000000                0.0, 1629,         timestamp   cmdb_id                                           kpi_name         value                   ts_utc8           p95            p5  high_anom  low_anom  high_dev  low_dev
0      1615384800      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09 2021-03-10 22:00:00+08:00  2.040136e+09  2.040136e+09       True      True  0.000000      0.0
1      1615384800      IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.140130e+01 2021-03-10 22:00:00+08:00  3.754270e+01  3.011800e+00      False     False       NaN      NaN
2      1615384800      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  7.284311e+08 2021-03-10 22:00:00+08:00  8.286598e+08  7.040201e+07      False     False       NaN      NaN
3      1615384800      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.792518e+07 2021-03-10 22:00:00+08:00  5.792518e+07  5.792518e+07       True      True  0.000000      0.0
4      1615384800      IG01          JVM-Operating System_7778_JVM_JVM_CPULoad  2.145000e-01 2021-03-10 22:00:00+08:00  2.145000e-01  1.661000e-01       True     False  0.000000      NaN
...           ...       ...                                                ...           ...                       ...           ...           ...        ...       ...       ...      ...
26301  1615386600  apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  4.000000e-01 2021-03-10 22:30:00+08:00  4.000000e-01  3.000000e-01       True     False  0.000000      NaN
26302  1615386600  apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  7.000000e+00 2021-03-10 22:30:00+08:00  7.000000e+00  7.000000e+00       True      True  0.000000      0.0
26303  1615386600  apache02                       OSLinux-OSLinux_SWAP_SWAP_Si  0.000000e+00 2021-03-10 22:30:00+08:00  0.000000e+00  0.000000e+00       True      True  0.000000      0.0
26304  1615386600  apache02                       OSLinux-OSLinux_SWAP_SWAP_So  0.000000e+00 2021-03-10 22:30:00+08:00  0.000000e+00  0.000000e+00       True      True  0.000000      0.0
26305  1615386600  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.585253e+06 2021-03-10 22:30:00+08:00  6.585162e+06  6.583538e+06       True     False  0.000014      NaN

[26306 rows x 11 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse in_window_marked from kernel (should exist)
tz = pytz.timezone('Asia/Shanghai')
if 'in_window_marked' not in globals():
    raise RuntimeError("Expected in_window_marked in kernel from prior step.")

# Filter to rows that are anomalous (high or low)
anom_rows = in_window_marked[(in_window_marked['high_anom'] | in_window_marked['low_anom'])].copy()

# Ensure sorted
anom_rows = anom_rows.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

# Identify run boundaries: consecutive if timestamp diff == 60 seconds within each group
diff_secs = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].diff().fillna(9999)
anom_rows['new_run_flag'] = (diff_secs != 60).astype(int)
# run identifier per group
anom_rows['run_id'] = anom_rows.groupby(['cmdb_id','kpi_name'])['new_run_flag'].cumsum()

# Aggregate runs into faults
group_cols = ['cmdb_id','kpi_name','run_id']
agg = anom_rows.groupby(group_cols, as_index=False).agg(
    fault_start_ts = ('ts_utc8', 'min'),
    fault_end_ts   = ('ts_utc8', 'max'),
    points         = ('timestamp', 'count'),
    high_count     = ('high_anom', 'sum'),
    low_count      = ('low_anom', 'sum'),
    max_high_deviation = ('high_dev', 'max'),
    max_low_deviation  = ('low_dev', 'max'),
    max_value_in_run   = ('value', 'max'),
    min_value_in_run   = ('value', 'min'),
)

# Fill NaNs in deviation aggregations with 0
agg['max_high_deviation'] = agg['max_high_deviation'].fillna(0.0)
agg['max_low_deviation']  = agg['max_low_deviation'].fillna(0.0)

# Determine fault_type by majority
def decide_type(row):
    if row['high_count'] > row['low_count']:
        return 'high'
    elif row['low_count'] > row['high_count']:
        return 'low'
    else:
        return 'mixed'

agg['fault_type'] = agg.apply(decide_type, axis=1)

# duration_minutes (inclusive) equals points when consecutive by 60s
agg['duration_minutes'] = agg['points']

# Severity metric for sorting
agg['severity'] = agg[['max_high_deviation','max_low_deviation']].max(axis=1)

# Format timestamps for compact display
agg['fault_start_ts_str'] = agg['fault_start_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
agg['fault_end_ts_str']   = agg['fault_end_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Build final faults_table and keep full table in kernel variable
faults_table = agg[[
    'cmdb_id','kpi_name','fault_start_ts','fault_end_ts','fault_start_ts_str','fault_end_ts_str',
    'duration_minutes','points','fault_type','max_high_deviation','max_low_deviation',
    'max_value_in_run','min_value_in_run','high_count','low_count','severity'
]].copy()

# Compact outputs:
cols_out = ['cmdb_id','kpi_name','fault_start_ts_str','fault_end_ts_str','duration_minutes','points','fault_type','max_high_deviation','max_low_deviation']

# A) Top 20 faults by earliest start
top_by_start = faults_table.sort_values('fault_start_ts').head(20)[cols_out].reset_index(drop=True)

# B) Top 20 faults by severity desc
top_by_severity = faults_table.sort_values('severity', ascending=False).head(20)[cols_out].reset_index(drop=True)

# C) Summary: total faults and distinct components with faults
total_faults = int(faults_table.shape[0])
distinct_components_with_faults = int(faults_table['cmdb_id'].nunique())
summary = pd.DataFrame({
    'total_faults':[total_faults],
    'distinct_components_with_faults':[distinct_components_with_faults]
})

# Display compact results (as variables)
top_by_start, top_by_severity, summary

```
Out[4]:
```
```python
# Plain-English summary built from kernel variables: top_by_start, top_by_severity, summary, faults_table
# Outputs a single compact string.

# Prepare helper to format example rows
def fmt_rows(df, n=5):
    if df is None or len(df)==0:
        return "  (none)"
    lines = []
    for r in df.head(n).itertuples(index=False):
        lines.append(f"  - {r.cmdb_id} | {r.kpi_name} | start={r.fault_start_ts_str} | end={r.fault_end_ts_str} | pts={int(r.points)} | type={r.fault_type} | max_high_dev={float(r.max_high_deviation):.6g} | max_low_dev={float(r.max_low_deviation):.6g}")
    return "\n".join(lines)

# Extract totals
if 'summary' in globals() and not summary.empty:
    total_faults = int(summary.loc[0,'total_faults'])
    distinct_components = int(summary.loc[0,'distinct_components_with_faults'])
else:
    total_faults = None
    distinct_components = None

summary_text = (
    "Fault-run detection summary (incident window 2021-03-10 22:00—22:30 UTC+8):\n\n"
    f"- Total faults detected: {total_faults}\n"
    f"- Distinct components with faults: {distinct_components}\n\n"
    "- Observations:\n"
    "  * Many faults are short (1–2 minutes) across many KPIs; a subset are long-running sequences.\n"
    "  * Example earliest faults (top by start time):\n"
    f"{fmt_rows(top_by_start, 5)}\n\n"
    "  * Example most severe faults (top by deviation):\n"
    f"{fmt_rows(top_by_severity, 5)}\n\n"
    "- Notable specifics:\n"
    "  * IG01 shows a long-running fault for JVM memory (HeapMemoryMax / NoHeapMemoryUsed) spanning the full window (22:00–22:30, 31 minutes).\n"
    "  * Many MySQL-related metrics (Mysql01/Mysql02) show the largest relative deviations (severity examples with max_high_deviation ≈ 19.0), primarily in short 1–2 minute faults around 22:25–22:26.\n"
    "  * The majority of faults are distributed across all 14 components monitored.\n\n"
    "Stored object: the full faults table is available in the kernel as 'faults_table' for further inspection.\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name       fault_start_ts_str         fault_end_ts_str  duration_minutes  points fault_type  max_high_deviation  max_low_deviation
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2021-03-10 22:00:00 CST  2021-03-10 22:30:00 CST                31      31      mixed            0.000000       0.000000e+00
1    Mysql02  OSLinux-OSLinux_FILESYSTEM_-mysqlbak_FSInodeUs...  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1        low            0.000000       0.000000e+00
2   apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  2021-03-10 22:00:00 CST  2021-03-10 22:01:00 CST                 2       2        low            0.000000       1.382236e-05
3    Mysql02    OSLinux-OSLinux_FILESYSTEM_-mysqlbak_FSCapacity  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1      mixed            0.000000       0.000000e+00
4    Mysql02  OSLinux-OSLinux_FILESYSTEM_-mysql_FSInodeUsedP...  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1        low            0.000000       3.995365e-05
5    Mysql02       OSLinux-OSLinux_FILESYSTEM_-mysql_FSCapacity  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1        low            0.000000       3.143189e-06
6    Mysql02  OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPe...  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1        low            0.000000       0.000000e+00
7    Mysql02        OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1        low            0.000000       1.253243e-05
8    Mysql02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1       high            0.000000       0.000000e+00
9    Mysql02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1       high            0.000000       0.000000e+00
10      IG02  OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPe...  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1       high            0.000000       0.000000e+00
11   Mysql02  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1       high            0.000000       0.000000e+00
12   Mysql02        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1       high            0.000000       0.000000e+00
13   Mysql02    OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1        low            0.000000       0.000000e+00
14   Mysql02            OSLinux-OSLinux_FILESYSTEM_-_FSCapacity  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1        low            0.000000       3.816590e-07
15   Mysql02                    OSLinux-CPU_CPU-0_SingleCpuidle  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1       high            0.016945       0.000000e+00
16   Mysql02         Mysql-MySQL_3306_max trx lock memory bytes  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1        low            0.000000       0.000000e+00
17   Mysql02                    Mysql-MySQL_3306_ThreadsRunning  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1        low            0.000000       0.000000e+00
18   Mysql02                   Mysql-MySQL_3306_Threads Created  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1      mixed            0.000000       0.000000e+00
19   Mysql02                 Mysql-MySQL_3306_Tc log page waits  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1      mixed            0.000000       0.000000e+00,      cmdb_id                                         kpi_name       fault_start_ts_str         fault_end_ts_str  duration_minutes  points fault_type  max_high_deviation  max_low_deviation
0    Mysql02                Mysql-MySQL_3306_Innodb data read  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
1    Mysql02               Mysql-MySQL_3306_Innodb pages read  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
2    Mysql01            Mysql-MySQL_3306_Innodb pages created  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1       high           19.000000                0.0
3    Mysql02               Mysql-MySQL_3306_Innodb data reads  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
4    Mysql02                 Mysql-MySQL_3306_Aborted Clients  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
5    Mysql02             Mysql-MySQL_3306_Max trx rows locked  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
6       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-10 22:17:00 CST  2021-03-10 22:18:00 CST                 2       2      mixed           19.000000                0.0
7   Tomcat02   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-10 22:29:00 CST  2021-03-10 22:30:00 CST                 2       2      mixed           19.000000                0.0
8    Mysql02        Mysql-MySQL_3306_Opened table definitions  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
9    Mysql01                      Mysql-MySQL_3306_Com Insert  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1       high           19.000000                0.0
10   Mysql01           Mysql-MySQL_3306_Innodb Row Lock Waits  2021-03-10 22:13:00 CST  2021-03-10 22:14:00 CST                 2       2      mixed           19.000000                0.0
11   Mysql01            Mysql-MySQL_3306_Innodb Row Lock Time  2021-03-10 22:13:00 CST  2021-03-10 22:14:00 CST                 2       2      mixed           19.000000                0.0
12   Mysql02        Mysql-MySQL_3306_Innodb buffer pool reads  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
13   Mysql02                Mysql-MySQL_3306_Handler Read Rnd  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
14   Mysql02                      Mysql-MySQL_3306_Sort Range  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
15   Mysql02               Mysql-MySQL_3306_Handler Read Next  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed           19.000000                0.0
16   Mysql02         Mysql-MySQL_3306_Table open cache misses  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed            9.384498                0.0
17   Mysql02                   Mysql-MySQL_3306_Opened Tables  2021-03-10 22:25:00 CST  2021-03-10 22:26:00 CST                 2       2      mixed            9.384498                0.0
18   Mysql02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  2021-03-10 22:16:00 CST  2021-03-10 22:16:00 CST                 1       1       high            3.000000                0.0
19   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead  2021-03-10 22:00:00 CST  2021-03-10 22:00:00 CST                 1       1       high            3.000000                0.0,    total_faults  distinct_components_with_faults
0         10944                               14)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df and log_df from kernel
# Ensure ts_utc8 exists for both
for df in (trace_df, log_df):
    if 'ts_utc8' not in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        df['ts_utc8'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# Components lists
components_all = [
    'apache01','apache02',
    'Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02',
    'IG01','IG02',
    'Mysql01','Mysql02',
    'Redis01','Redis02'
]
log_components = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04','Mysql01','Mysql02']

# Incident window (inclusive) in UTC+8
win_start = pd.Timestamp('2021-03-10 22:00:00').tz_localize(tz)
win_end   = pd.Timestamp('2021-03-10 22:30:00').tz_localize(tz)

# 1) Trace analysis
trace_win = trace_df[
    trace_df['cmdb_id'].isin(components_all) &
    (trace_df['ts_utc8'] >= win_start) &
    (trace_df['ts_utc8'] <= win_end)
].copy()

# Aggregate per trace_name
def p95_series(s):
    s2 = s.dropna()
    return float(np.nanpercentile(s2,95)) if s2.size>0 else np.nan

trace_aggs = (
    trace_win
    .groupby('trace_name', as_index=False)
    .agg(
        count=('value','count'),
        mean=('value','mean'),
        max=('value','max'),
        p95=('value', p95_series),
        earliest_ts=('ts_utc8','min')
    )
)

# Format earliest_ts for compact output
trace_aggs['earliest_timestamp_in_window'] = trace_aggs['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
trace_aggs_top20 = trace_aggs.sort_values('max', ascending=False).head(20)[
    ['trace_name','count','mean','max','p95','earliest_timestamp_in_window']
].reset_index(drop=True)

# store full filtered trace_aggs in kernel variable for follow-up
trace_aggs_full = trace_aggs.copy()

# 2) Log analysis
log_win = log_df[
    log_df['cmdb_id'].isin(log_components) &
    (log_df['ts_utc8'] >= win_start) &
    (log_df['ts_utc8'] <= win_end)
].copy()

# Basic counts per (cmdb_id, log_name)
log_grp = log_win.groupby(['cmdb_id','log_name'], as_index=False).agg(
    count=('value','count')
)

# sum_error_count per (cmdb_id, log_name) only meaningful when log_name == 'log.error_count'
err_sum = (
    log_win[log_win['log_name']=='log.error_count']
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(sum_error_count=('value','sum'))
)

# max_row_count per (cmdb_id, log_name) meaningful when log_name == 'log.row_count'
rowmax = (
    log_win[log_win['log_name']=='log.row_count']
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(max_row_count=('value','max'))
)

# earliest_error_ts where log_name=='log.error_count' and value>0
err_ts = (
    log_win[(log_win['log_name']=='log.error_count') & (log_win['value']>0)]
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(earliest_error_ts=('ts_utc8','min'))
)
if not err_ts.empty:
    err_ts['earliest_error_ts'] = err_ts['earliest_error_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    err_ts = err_ts[['cmdb_id','log_name','earliest_error_ts']]
else:
    # empty frame with right cols
    err_ts = pd.DataFrame(columns=['cmdb_id','log_name','earliest_error_ts'])

# Merge aggregates
log_aggs = log_grp.merge(err_sum, on=['cmdb_id','log_name'], how='left')
log_aggs = log_aggs.merge(rowmax, on=['cmdb_id','log_name'], how='left')
log_aggs = log_aggs.merge(err_ts, on=['cmdb_id','log_name'], how='left')

# Fill NaNs for numeric aggregates with 0 and leave earliest_error_ts as None where missing
log_aggs['sum_error_count'] = log_aggs['sum_error_count'].fillna(0)
log_aggs['max_row_count'] = log_aggs['max_row_count'].fillna(0)
log_aggs['earliest_error_ts'] = log_aggs['earliest_error_ts'].where(pd.notnull(log_aggs['earliest_error_ts']), None)

# Prepare compact table (limit 20 rows)
log_aggs_compact = log_aggs.sort_values(['cmdb_id','log_name']).head(20)[
    ['cmdb_id','log_name','count','sum_error_count','max_row_count','earliest_error_ts']
].reset_index(drop=True)

# store full aggregates for follow-up
trace_aggs, trace_aggs_full, log_aggs_compact, log_aggs

```
Out[5]:
```
```python
# Build a concise plain-English summary using existing kernel aggregates:
# Assumes variables from prior steps exist: trace_aggs_top20, trace_aggs_full, log_aggs_compact, log_aggs

def fmt_top_trace(df, n=5):
    if df is None or df.shape[0]==0:
        return "  (no traces)"
    lines = []
    for r in df.head(n).itertuples(index=False):
        lines.append(f"  - {r.trace_name} | cnt={int(r.count)} | max={r.max:.6g} | p95={r.p95:.6g} | first={r.earliest_timestamp_in_window}")
    return "\n".join(lines)

def fmt_top_logs(df, n=10):
    if df is None or df.shape[0]==0:
        return "  (no logs)"
    lines = []
    for r in df.head(n).itertuples(index=False):
        lines.append(f"  - {r.cmdb_id} | {r.log_name} | cnt={int(r.count)} | sum_error={float(r.sum_error_count):.0f} | max_row={float(r.max_row_count):.0f} | first_err={r.earliest_error_ts}")
    return "\n".join(lines)

# Basic checks and counts
num_trace_names = int(trace_aggs_full.shape[0]) if 'trace_aggs_full' in globals() else 0
num_log_rows = int(log_aggs.shape[0]) if 'log_aggs' in globals() else 0

# Top trace by max value (from trace_aggs_top20 if available)
top_trace_example = ""
if 'trace_aggs_top20' in globals() and trace_aggs_top20.shape[0]>0:
    t0 = trace_aggs_top20.iloc[0]
    top_trace_example = f"{t0.trace_name} (max={t0['max']:.6g}, p95={t0['p95']:.6g}, count={int(t0['count'])}, first={t0.earliest_timestamp_in_window})"

# Logs: check if any error counts > 0
errors_present = False
first_error_details = None
if 'log_aggs' in globals() and not log_aggs.empty:
    if (log_aggs['sum_error_count'] > 0).any():
        errors_present = True
        row = log_aggs[log_aggs['sum_error_count']>0].iloc[0]
        first_error_details = (row['cmdb_id'], row['log_name'], row.get('earliest_error_ts', None))

summary_lines = [
    "Trace & Log correlation summary for incident window 2021-03-10 22:00—22:30 (UTC+8):",
    "",
    f"- Trace data: analyzed {num_trace_names} distinct trace_name aggregates in-window.",
    f"- Top trace by peak latency: {top_trace_example}" if top_trace_example else "  (no top trace available)",
    "",
    "Top trace_name examples (by peak value):",
    fmt_top_trace(trace_aggs_top20, n=6),
    "",
    "- Interpretation (trace):",
    "  * Many trace entries show elevated max/p95 latencies (examples include traces from IG01).",
    "  * Row-count trace metrics (e.g., trace.*.row_count) indicate sustained traffic during the window.",
    "",
    f"- Log data: analyzed {num_log_rows} (cmdb_id,log_name) groups for Tomcat/MySQL.",
    "Top log aggregates (sample):",
    fmt_top_logs(log_aggs_compact, n=12),
    "",
]

if errors_present:
    summary_lines += [
        f"- Error logs: positive log.error_count found: example {first_error_details}.",
        "  * Earliest error timestamps and sums are available in the log aggregates for further triage."
    ]
else:
    summary_lines += [
        "- Error logs: no log.error_count > 0 observed in the filtered Tomcat/MySQL logs during the window.",
        "  * log.row_count values are present (indicating request volumes), but sum_error_count is zero for the sampled components."
    ]

summary_lines += [
    "",
    "Overall conclusion:",
    " - Trace data shows latency spikes and sustained traffic for services (notably traces involving IG01), which can explain some metric anomalies.",
    " - Application logs (Tomcat/MySQL) in the sampled window show request volume but no recorded log.error_count > 0, so direct error-log evidence is absent in these aggregates.",
    "",
    "Stored objects for follow-up: 'trace_aggs_full' and 'log_aggs' contain the full aggregates."
]

summary_text = "\n".join(summary_lines)

# Return the compact summary string as the cell output
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         trace_name  count       mean        max        p95               earliest_ts earliest_timestamp_in_window
0     trace.from_IG01.duration_mean    120   0.133212   0.327687   0.228156 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
1      trace.from_IG01.duration_p95    120   0.756657   1.286400   1.132665 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
2         trace.from_IG01.row_count    120  61.566667  89.000000  78.000000 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
3     trace.from_IG02.duration_mean    120   0.127804   0.251197   0.222722 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
4      trace.from_IG02.duration_p95    120   0.732366   1.166800   1.092402 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
..                              ...    ...        ...        ...        ...                       ...                          ...
61   trace.to_dockerB1.duration_p95     60   0.863131   1.478950   1.252092 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
62      trace.to_dockerB1.row_count     60  55.766667  96.000000  72.150000 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
63  trace.to_dockerB2.duration_mean     60   0.104352   0.243982   0.192293 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
64   trace.to_dockerB2.duration_p95     60   0.652303   1.526400   1.113882 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
65      trace.to_dockerB2.row_count     60  57.133333  97.000000  83.300000 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST

[66 rows x 7 columns],                          trace_name  count       mean        max        p95               earliest_ts earliest_timestamp_in_window
0     trace.from_IG01.duration_mean    120   0.133212   0.327687   0.228156 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
1      trace.from_IG01.duration_p95    120   0.756657   1.286400   1.132665 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
2         trace.from_IG01.row_count    120  61.566667  89.000000  78.000000 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
3     trace.from_IG02.duration_mean    120   0.127804   0.251197   0.222722 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
4      trace.from_IG02.duration_p95    120   0.732366   1.166800   1.092402 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
..                              ...    ...        ...        ...        ...                       ...                          ...
61   trace.to_dockerB1.duration_p95     60   0.863131   1.478950   1.252092 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
62      trace.to_dockerB1.row_count     60  55.766667  96.000000  72.150000 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
63  trace.to_dockerB2.duration_mean     60   0.104352   0.243982   0.192293 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
64   trace.to_dockerB2.duration_p95     60   0.652303   1.526400   1.113882 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST
65      trace.to_dockerB2.row_count     60  57.133333  97.000000  83.300000 2021-03-10 22:00:00+08:00      2021-03-10 22:00:00 CST

[66 rows x 7 columns],      cmdb_id          log_name  count  sum_error_count  max_row_count earliest_error_ts
0    Mysql01   log.error_count     16              0.0            0.0              None
1    Mysql01     log.row_count     16              0.0            4.0              None
2   Tomcat01   log.error_count     31              0.0            0.0              None
3   Tomcat01  log.gc_pause_max      8              0.0            0.0              None
4   Tomcat01     log.row_count     31              0.0          152.0              None
5   Tomcat02   log.error_count     30              0.0            0.0              None
6   Tomcat02  log.gc_pause_max     13              0.0            0.0              None
7   Tomcat02     log.row_count     30              0.0          154.0              None
8   Tomcat03   log.error_count     31              0.0            0.0              None
9   Tomcat03  log.gc_pause_max      8              0.0            0.0              None
10  Tomcat03     log.row_count     31              0.0          152.0              None
11  Tomcat04   log.error_count     31              0.0            0.0              None
12  Tomcat04  log.gc_pause_max      7              0.0            0.0              None
13  Tomcat04     log.row_count     31              0.0          166.0              None,      cmdb_id          log_name  count  sum_error_count  max_row_count earliest_error_ts
0    Mysql01   log.error_count     16              0.0            0.0              None
1    Mysql01     log.row_count     16              0.0            4.0              None
2   Tomcat01   log.error_count     31              0.0            0.0              None
3   Tomcat01  log.gc_pause_max      8              0.0            0.0              None
4   Tomcat01     log.row_count     31              0.0          152.0              None
5   Tomcat02   log.error_count     30              0.0            0.0              None
6   Tomcat02  log.gc_pause_max     13              0.0            0.0              None
7   Tomcat02     log.row_count     30              0.0          154.0              None
8   Tomcat03   log.error_count     31              0.0            0.0              None
9   Tomcat03  log.gc_pause_max      8              0.0            0.0              None
10  Tomcat03     log.row_count     31              0.0          152.0              None
11  Tomcat04   log.error_count     31              0.0            0.0              None
12  Tomcat04  log.gc_pause_max      7              0.0            0.0              None
13  Tomcat04     log.row_count     31              0.0          166.0              None)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse faults_table from kernel
if 'faults_table' not in globals():
    raise RuntimeError("Expected 'faults_table' in kernel.")

# Ensure fault_start_ts is timezone-aware datetime in UTC+8
fts = faults_table.copy()
if fts['fault_start_ts'].dtype == object:
    fts['fault_start_ts'] = pd.to_datetime(fts['fault_start_ts']).dt.tz_localize(tz)
else:
    # ensure tz
    if fts['fault_start_ts'].dt.tz is None:
        fts['fault_start_ts'] = fts['fault_start_ts'].dt.tz_localize(tz)
    else:
        fts['fault_start_ts'] = fts['fault_start_ts'].dt.tz_convert(tz)

# Use the minute as-is (fault_start_ts truncated to minute should already be minute-aligned)
# Create string representation for display
fts['fault_start_min_str'] = fts['fault_start_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Severity already in 'severity' column as max(max_high_deviation, max_low_deviation)
# Aggregate per fault_start minute
def sample_examples(subdf, n=3):
    ex = subdf.head(n).apply(
        lambda r: f"{r['cmdb_id']} | {r['kpi_name']} | {r['fault_type']} | {float(r['max_high_deviation']):.6g}|{float(r['max_low_deviation']):.6g}",
        axis=1
    ).tolist()
    return " ; ".join(ex)

grp = (
    fts
    .groupby(['fault_start_min_str'], sort=False)
    .agg(
        faults_starting_count=('cmdb_id','size'),
        distinct_components_with_faults_starting=('cmdb_id', pd.Series.nunique),
        max_severity_this_minute=('severity','max')
    )
    .reset_index()
)

# Add example samples by mapping group
examples = fts.groupby('fault_start_min_str').apply(lambda d: sample_examples(d.sort_values(['severity'], ascending=False), n=3)).reset_index(name='example_sample')

faults_by_minute = grp.merge(examples, on='fault_start_min_str', how='left')

# Sort by faults_starting_count desc and limit 20
faults_by_minute = faults_by_minute.sort_values('faults_starting_count', ascending=False).reset_index(drop=True)
top20_faults_by_minute = faults_by_minute.head(20)[[
    'fault_start_min_str','faults_starting_count','distinct_components_with_faults_starting','max_severity_this_minute','example_sample'
]].rename(columns={'fault_start_min_str':'fault_start_ts'})

# Top 2 minutes separately as tiny list (keep as DataFrame of 2 rows)
top2_minutes = top20_faults_by_minute.head(2).reset_index(drop=True)

# Return compact outputs (variables)
faults_by_minute, top20_faults_by_minute, top2_minutes

```
Out[6]:
```
```python
# Build a concise plain-English summary from 'faults_by_minute' / 'top20_faults_by_minute' / 'top2_minutes'
# Assumes these variables exist in the kernel from prior steps.

def row_to_example(r):
    return f"{r.fault_start_ts} | count={int(r.faults_starting_count)} | distinct_components={int(r.distinct_components_with_faults_starting)} | max_severity={float(r.max_severity_this_minute):.6g}"

# Prepare top-2 minute summaries if available
if 'top2_minutes' in globals() and not top2_minutes.empty:
    top2_lines = []
    for r in top2_minutes.itertuples(index=False):
        top2_lines.append(row_to_example(r))
    top2_text = "\n".join(f"  {i+1}. {line}" for i,line in enumerate(top2_lines))
else:
    top2_text = "  (no top minutes available)"

# Prepare a few example rows from top20 for illustration
if 'top20_faults_by_minute' in globals() and not top20_faults_by_minute.empty:
    examples = []
    for r in top20_faults_by_minute.head(5).itertuples(index=False):
        ex = f"{r.fault_start_ts} -> faults={int(r.faults_starting_count)}, comps={int(r.distinct_components_with_faults_starting)}, max_sev={float(r.max_severity_this_minute):.6g}"
        examples.append("  - " + ex)
    examples_text = "\n".join(examples)
else:
    examples_text = "  (no examples)"

summary = (
    "Per-minute fault-start summary (incident window 2021-03-10 22:00—22:30 UTC+8):\n\n"
    f"- Top minutes (by number of faults starting):\n{top2_text}\n\n"
    "- Notes:\n"
    "  * The busiest minute is 2021-03-10 22:00:00 (UTC+8) with 607 faults starting across 14 components; max severity = 19.0.\n"
    "  * The second busiest is 2021-03-10 22:02:00 (UTC+8) with 524 faults starting across 14 components; max severity ≈ 0.339754.\n"
    "  * Many minutes show faults across all 14 monitored components; example top-minute rows:\n"
    f"{examples_text}\n\n"
    "Stored object: the per-minute faults summary is available in the kernel as 'faults_by_minute'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        fault_start_min_str  faults_starting_count  distinct_components_with_faults_starting  max_severity_this_minute                                     example_sample
0   2021-03-10 22:00:00 CST                    607                                        14                 19.000000  Mysql01 | Mysql-MySQL_3306_Com Insert | high |...
1   2021-03-10 22:02:00 CST                    524                                        14                  0.339754  IG02 | JVM-Memory_7778_JVM_Memory_HeapMemoryUs...
2   2021-03-10 22:11:00 CST                    482                                        14                  0.666667  Tomcat03 | OSLinux-OSLinux_LOCALDISK_LOCALDISK...
3   2021-03-10 22:08:00 CST                    473                                        14                  0.818182  Tomcat03 | OSLinux-OSLinux_LOCALDISK_LOCALDISK...
4   2021-03-10 22:05:00 CST                    451                                        14                  0.875000  IG01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb...
..                      ...                    ...                                       ...                       ...                                                ...
26  2021-03-10 22:04:00 CST                    263                                        14                  2.951613  IG02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb...
27  2021-03-10 22:07:00 CST                    260                                        14                  0.666667  MG01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda...
28  2021-03-10 22:03:00 CST                    249                                        14                  1.808989  IG02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb...
29  2021-03-10 22:12:00 CST                    249                                        14                  0.818182  Tomcat03 | OSLinux-OSLinux_LOCALDISK_LOCALDISK...
30  2021-03-10 22:10:00 CST                    213                                        14                  0.033664  IG02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb...

[31 rows x 5 columns],              fault_start_ts  faults_starting_count  distinct_components_with_faults_starting  max_severity_this_minute                                     example_sample
0   2021-03-10 22:00:00 CST                    607                                        14                 19.000000  Mysql01 | Mysql-MySQL_3306_Com Insert | high |...
1   2021-03-10 22:02:00 CST                    524                                        14                  0.339754  IG02 | JVM-Memory_7778_JVM_Memory_HeapMemoryUs...
2   2021-03-10 22:11:00 CST                    482                                        14                  0.666667  Tomcat03 | OSLinux-OSLinux_LOCALDISK_LOCALDISK...
3   2021-03-10 22:08:00 CST                    473                                        14                  0.818182  Tomcat03 | OSLinux-OSLinux_LOCALDISK_LOCALDISK...
4   2021-03-10 22:05:00 CST                    451                                        14                  0.875000  IG01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb...
5   2021-03-10 22:28:00 CST                    438                                        14                  2.704180  IG01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda...
6   2021-03-10 22:01:00 CST                    437                                        14                  1.261421  Tomcat04 | OSLinux-CPU_CPU_CPUWio | high | 1.2...
7   2021-03-10 22:22:00 CST                    392                                        14                  2.050679  Mysql02 | Mysql-MySQL_3306_Innodb Row Lock Tim...
8   2021-03-10 22:26:00 CST                    381                                        14                  0.388889  Tomcat03 | OSLinux-OSLinux_LOCALDISK_LOCALDISK...
9   2021-03-10 22:13:00 CST                    379                                        14                 19.000000  Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Wai...
10  2021-03-10 22:21:00 CST                    371                                        14                  1.083333  apache02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK...
11  2021-03-10 22:16:00 CST                    348                                        14                  3.000000  Mysql02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-...
12  2021-03-10 22:27:00 CST                    347                                        14                  2.026316  Mysql01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-...
13  2021-03-10 22:15:00 CST                    344                                        14                  1.857143  apache02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK...
14  2021-03-10 22:20:00 CST                    341                                        14                  0.428571  MG02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb...
15  2021-03-10 22:19:00 CST                    337                                        14                  0.436234  Redis01 | OSLinux-CPU_CPU_CPUWio | high | 0.43...
16  2021-03-10 22:25:00 CST                    337                                        14                 19.000000  Mysql02 | Mysql-MySQL_3306_Innodb data read | ...
17  2021-03-10 22:30:00 CST                    336                                        14                  0.879245  IG01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda...
18  2021-03-10 22:14:00 CST                    325                                        14                  1.667070  Redis01 | OSLinux-CPU_CPU_CPUWio | high | 1.66...
19  2021-03-10 22:24:00 CST                    320                                        14                  0.583090  Redis02 | OSLinux-CPU_CPU_CPUWio | high | 0.58...,             fault_start_ts  faults_starting_count  distinct_components_with_faults_starting  max_severity_this_minute                                     example_sample
0  2021-03-10 22:00:00 CST                    607                                        14                 19.000000  Mysql01 | Mysql-MySQL_3306_Com Insert | high |...
1  2021-03-10 22:02:00 CST                    524                                        14                  0.339754  IG02 | JVM-Memory_7778_JVM_Memory_HeapMemoryUs...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse kernel vars: top2_minutes, faults_table
if 'top2_minutes' not in globals() or 'faults_table' not in globals():
    raise RuntimeError("Expected 'top2_minutes' and 'faults_table' in kernel.")

# Determine minute column name in top2_minutes
if 'fault_start_ts' in top2_minutes.columns:
    minute_col = 'fault_start_ts'
elif 'fault_start_min_str' in top2_minutes.columns:
    minute_col = 'fault_start_min_str'
else:
    # fallback to any column that contains '2021-03-10'
    candidates = [c for c in top2_minutes.columns if top2_minutes[c].astype(str).str.contains('2021-03-10').any()]
    if not candidates:
        raise RuntimeError("Couldn't find minute column in top2_minutes.")
    minute_col = candidates[0]

minutes = top2_minutes[minute_col].astype(str).tolist()

# Work on a copy
fts = faults_table.copy()

# Ensure string timestamp column exists
if 'fault_start_ts_str' not in fts.columns:
    # create from fault_start_ts
    tz = pytz.timezone('Asia/Shanghai')
    fts['fault_start_ts_str'] = pd.to_datetime(fts['fault_start_ts']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

rows_list = []
top3_records = []

for minute in minutes:
    sub = fts[fts['fault_start_ts_str'] == minute].copy()
    if sub.empty:
        continue
    # Aggregate per component
    grp = sub.groupby('cmdb_id', as_index=False).agg(
        faults_count=('cmdb_id','size'),
        max_severity=('severity','max'),
        earliest_fault_start_ts_dt=('fault_start_ts','min')
    )
    # format earliest ts
    tz = pytz.timezone('Asia/Shanghai')
    grp['earliest_fault_start_ts'] = pd.to_datetime(grp['earliest_fault_start_ts_dt']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # example_kpis: up to 3 distinct kpi_name with highest severity for that component in this minute
    def top_kpis_for_component(df_comp, n=3):
        # sort by severity desc, take distinct kpi_name
        kp = df_comp.sort_values('severity', ascending=False)['kpi_name'].drop_duplicates().head(n).tolist()
        return ", ".join(kp)
    examples = []
    for cid in grp['cmdb_id']:
        comp_sub = sub[sub['cmdb_id']==cid]
        examples.append(top_kpis_for_component(comp_sub, n=3))
    grp['example_kpis'] = examples
    grp['fault_minute'] = minute
    # select and order columns
    grp_out = grp[['fault_minute','cmdb_id','faults_count','max_severity','earliest_fault_start_ts','example_kpis']].copy()
    # sort per minute by faults_count desc then max_severity desc, limit top 10
    grp_out = grp_out.sort_values(['faults_count','max_severity'], ascending=[False, False]).head(10).reset_index(drop=True)
    rows_list.append(grp_out)
    # top 3 candidate components for this minute
    top3 = grp_out.head(3)['cmdb_id'].tolist()
    top3_records.append({'fault_minute': minute, 'top3_components': ", ".join(top3)})

# Concatenate results
if rows_list:
    candidate_components_per_minute = pd.concat(rows_list, ignore_index=True)
else:
    candidate_components_per_minute = pd.DataFrame(columns=['fault_minute','cmdb_id','faults_count','max_severity','earliest_fault_start_ts','example_kpis'])

top3_df = pd.DataFrame(top3_records)

# Return compact outputs for inspection
candidate_components_per_minute.head(20), top3_df, candidate_components_per_minute.shape

```
Out[7]:
```
```python
# Build a concise plain-English summary from existing kernel variables:
# Expects 'candidate_components_per_minute' and 'top3_df' to exist.

try:
    c = candidate_components_per_minute.copy()
    t3 = top3_df.copy()
except NameError as e:
    summary = "Required kernel variables ('candidate_components_per_minute' and 'top3_df') not found."
else:
    minutes = t3['fault_minute'].tolist()
    lines = []
    lines.append("Per-minute candidate components summary for the two top minutes (compact):")
    lines.append("")
    for minute in minutes:
        lines.append(f"- Minute: {minute}")
        sub = c[c['fault_minute'] == minute].sort_values(['faults_count','max_severity'], ascending=[False, False]).head(10)
        if sub.empty:
            lines.append("    (no faults for this minute)")
            continue
        # show up to top 5 components for readability
        for r in sub.head(5).itertuples(index=False):
            lines.append(f"    • {r.cmdb_id}: faults={int(r.faults_count)}, max_sev={float(r.max_severity):.6g}, earliest={r.earliest_fault_start_ts}, examples=[{r.example_kpis}]")
        # show top-3 candidates extracted earlier
        top3_row = t3[t3['fault_minute']==minute]
        top3_list = top3_row['top3_components'].iloc[0] if not top3_row.empty else "(none)"
        lines.append(f"    Top-3 candidate components (by faults_count, then severity): {top3_list}")
        lines.append("")
    # High-level conclusion
    lines.append("High-level conclusions:")
    lines.append(" - For 2021-03-10 22:00:00, Mysql01 and Mysql02 are the leading candidates (largest fault counts); Mysql01 shows the largest reported severity (max_severity ≈ 19.0).")
    lines.append(" - For 2021-03-10 22:02:00, Mysql02 and Mysql01 again lead in fault counts; other components (Redis02, IG02, Tomcat instances) also contributed notable faults.")
    lines.append(" - These two minutes show clustered multi-component faults across all monitored components; top candidates for follow-up are Mysql01 and Mysql02 first, then Redis02.")
    lines.append("")
    lines.append("Stored object: 'candidate_components_per_minute' contains the full per-minute per-component aggregates for follow-up.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               fault_minute   cmdb_id  faults_count  max_severity  earliest_fault_start_ts                                       example_kpis
0   2021-03-10 22:00:00 CST   Mysql01           137     19.000000  2021-03-10 22:00:00 CST  Mysql-MySQL_3306_Com Insert, Mysql-MySQL_3306_...
1   2021-03-10 22:00:00 CST   Mysql02           124      3.000000  2021-03-10 22:00:00 CST  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...
2   2021-03-10 22:00:00 CST   Redis02            54      3.000000  2021-03-10 22:00:00 CST  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...
3   2021-03-10 22:00:00 CST  apache01            47      0.200018  2021-03-10 22:00:00 CST  OSLinux-CPU_CPU_CPUWio, OSLinux-OSLinux_LOCALD...
4   2021-03-10 22:00:00 CST      IG02            37      0.142857  2021-03-10 22:00:00 CST  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
5   2021-03-10 22:00:00 CST   Redis01            37      0.061224  2021-03-10 22:00:00 CST  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUti...
6   2021-03-10 22:00:00 CST      IG01            35      0.014458  2021-03-10 22:00:00 CST  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNu...
7   2021-03-10 22:00:00 CST  Tomcat02            27      0.071578  2021-03-10 22:00:00 CST  OSLinux-CPU_CPU_CPULoad, OSLinux-OSLinux_NETWO...
8   2021-03-10 22:00:00 CST  Tomcat04            25      0.000403  2021-03-10 22:00:00 CST  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...
9   2021-03-10 22:00:00 CST  Tomcat03            25      0.000362  2021-03-10 22:00:00 CST  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...
10  2021-03-10 22:02:00 CST   Mysql02           140      0.200704  2021-03-10 22:02:00 CST  Mysql-MySQL_3306_Innodb data written, Mysql-My...
11  2021-03-10 22:02:00 CST   Mysql01           100      0.200000  2021-03-10 22:02:00 CST  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...
12  2021-03-10 22:02:00 CST   Redis02            47      0.206387  2021-03-10 22:02:00 CST  OSLinux-CPU_CPU_CPUSysTime, OSLinux-OSLinux_LO...
13  2021-03-10 22:02:00 CST      IG02            40      0.339754  2021-03-10 22:02:00 CST  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed, OSL...
14  2021-03-10 22:02:00 CST  Tomcat03            39      0.310345  2021-03-10 22:02:00 CST  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUti...
15  2021-03-10 22:02:00 CST      IG01            39      0.127243  2021-03-10 22:02:00 CST  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed, OSL...
16  2021-03-10 22:02:00 CST   Redis01            39      0.000004  2021-03-10 22:02:00 CST  OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity, O...
17  2021-03-10 22:02:00 CST  apache01            29      0.157149  2021-03-10 22:02:00 CST  OSLinux-CPU_CPU-3_SingleCpuUtil, OSLinux-OSLin...
18  2021-03-10 22:02:00 CST      MG02            15      0.100389  2021-03-10 22:02:00 CST  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNu...
19  2021-03-10 22:02:00 CST  apache02            10      0.066830  2021-03-10 22:02:00 CST  OSLinux-CPU_CPU_CPUSysTime, OSLinux-OSLinux_ME...,               fault_minute            top3_components
0  2021-03-10 22:00:00 CST  Mysql01, Mysql02, Redis02
1  2021-03-10 22:02:00 CST  Mysql02, Mysql01, Redis02, (20, 6))```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df and error_df from kernel; ensure ts_utc8 exists
for df in (trace_df, error_df):
    if 'ts_utc8' not in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        df['ts_utc8'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# Minutes and candidate lists
minute_strs = ['2021-03-10 22:00:00', '2021-03-10 22:02:00']
minutes = [pd.Timestamp(s).tz_localize(tz) for s in minute_strs]

candidates_by_minute = {
    '2021-03-10 22:00:00': ['Mysql01','Mysql02','Redis02'],
    '2021-03-10 22:02:00': ['Mysql02','Mysql01','Redis02']
}

# Helper p95
def p95_series(s):
    s2 = s.dropna()
    return float(np.nanpercentile(s2,95)) if s2.size>0 else np.nan

trace_rows = []
for minute in minutes:
    minute_str = minute.strftime('%Y-%m-%d %H:%M:%S %Z')
    cand_list = candidates_by_minute[minute.strftime('%Y-%m-%d %H:%M:%S')]
    # Filter trace_df to exact minute
    tfm = trace_df[trace_df['ts_utc8'] == minute]
    if tfm.empty:
        continue
    for cand in cand_list:
        # Condition: trace_name contains candidate OR (cmdb_id==candidate AND trace_name startswith 'trace.self')
        cond = tfm['trace_name'].str.contains(cand, na=False) | ((tfm['cmdb_id']==cand) & tfm['trace_name'].str.startswith('trace.self', na=False))
        sub = tfm[cond].copy()
        if sub.empty:
            continue
        agg = (
            sub.groupby('trace_name', as_index=False)
            .agg(
                count=('value','count'),
                mean=('value','mean'),
                max=('value','max'),
                p95=('value', p95_series),
                earliest_ts=('ts_utc8','min')
            )
        )
        agg['minute'] = minute_str
        agg['candidate_component'] = cand
        # format earliest_ts
        agg['earliest_timestamp_in_window'] = agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        trace_rows.append(agg[['minute','candidate_component','trace_name','count','mean','max','p95','earliest_timestamp_in_window']])

# Concatenate and sort, limit top 50 by minute then candidate then max desc
if trace_rows:
    candidate_traces_by_minute = pd.concat(trace_rows, ignore_index=True)
    candidate_traces_by_minute = candidate_traces_by_minute.sort_values(['minute','candidate_component','max'], ascending=[True,True,False])
    candidate_traces_by_minute = candidate_traces_by_minute.head(50).reset_index(drop=True)
else:
    candidate_traces_by_minute = pd.DataFrame(columns=['minute','candidate_component','trace_name','count','mean','max','p95','earliest_timestamp_in_window'])

# 2) Error log check for the two minutes and components [Mysql01, Mysql02, Redis02]
components_check = ['Mysql01','Mysql02','Redis02']
err_rows = error_df[ (error_df['ts_utc8'].isin(minutes)) & (error_df['cmdb_id'].isin(components_check)) ].copy()

if err_rows.empty:
    # explicit empty summary DataFrame
    candidate_error_logs_by_minute = pd.DataFrame(columns=['minute','cmdb_id','message_count','sample_messages'])
else:
    # format minute string
    err_rows['minute'] = err_rows['ts_utc8'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # aggregate per minute, cmdb_id
    def sample_msgs(s, n=5):
        return " || ".join(s.astype(str).dropna().unique()[:n])
    err_aggs = (
        err_rows.groupby(['minute','cmdb_id'], as_index=False)
        .agg(
            message_count=('message','count'),
            sample_messages=('message', lambda s: sample_msgs(s, n=5))
        )
    )
    candidate_error_logs_by_minute = err_aggs[['minute','cmdb_id','message_count','sample_messages']].reset_index(drop=True)

# Store results in kernel variables for follow-up (already named)
candidate_traces_by_minute, candidate_error_logs_by_minute

```
Out[8]:
```
```python
# Plain-English summary based on kernel results (compact).
# Uses existing kernel variables candidate_traces_by_minute and candidate_error_logs_by_minute.

rows_traces = 0
rows_errors = 0
if 'candidate_traces_by_minute' in globals():
    rows_traces = int(candidate_traces_by_minute.shape[0])
if 'candidate_error_logs_by_minute' in globals():
    rows_errors = int(candidate_error_logs_by_minute.shape[0])

summary = (
    "Result summary (compact):\n\n"
    f"- Trace detail: found {rows_traces} matching trace rows at the exact minutes 2021-03-10 22:00:00 and 22:02:00 (UTC+8) for the candidate components "
    "(Mysql01, Mysql02, Redis02). In other words, no per-minute trace entries matched the strict filters applied.\n\n"
    f"- Error logs: found {rows_errors} matching error_log rows at those exact minutes for Mysql01/Mysql02/Redis02. No error-log messages were captured at those exact timestamps.\n\n"
    "Interpretation & next steps:\n"
    " - There is no direct trace or error-log evidence at the exact minute timestamps to link metric faults to service-call spikes or logged errors for the listed candidates.\n"
    " - This absence can be due to timing granularity (events occur within the minute but timestamps differ by seconds), naming mismatches (trace_name patterns differ), or events appearing slightly before/after the exact minute.\n"
    " - Recommended follow-ups (brief):\n"
    "   1) Broaden the time window to ±1–2 minutes around each minute and re-run the candidate trace/error lookup.\n"
    "   2) Search trace_name patterns more broadly (e.g., 'to_', 'from_', component substrings, case-insensitive) and include traces where cmdb_id == candidate with 'trace.self'.\n"
    "   3) Inspect raw error logs over the full incident window (not only exact-minute timestamps) and examine log.csv for log.error_count or other error indicators.\n\n"
    "Stored objects for follow-up: 'candidate_traces_by_minute' and 'candidate_error_logs_by_minute' (both empty for the strict filters used)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [minute, candidate_component, trace_name, count, mean, max, p95, earliest_timestamp_in_window]
Index: [], Empty DataFrame
Columns: [minute, cmdb_id, message_count, sample_messages]
Index: [])```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df, log_df, error_df exist and have ts_utc8
for df in (trace_df, log_df, error_df):
    if 'ts_utc8' not in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        df['ts_utc8'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# Define base minutes and windows
base1 = pd.Timestamp('2021-03-10 22:00:00').tz_localize(tz)
base2 = pd.Timestamp('2021-03-10 22:02:00').tz_localize(tz)
window_A_minutes = [ (base1 + pd.Timedelta(minutes=i)).floor('min') for i in range(-2, 3) ]  # 21:58 .. 22:02
window_B_minutes = [ (base2 + pd.Timedelta(minutes=i)).floor('min') for i in range(-2, 3) ]  # 22:00 .. 22:04

# Map minute -> window labels
minute_to_windows = {}
for m in window_A_minutes:
    minute_to_windows.setdefault(m, []).append('A')
for m in window_B_minutes:
    minute_to_windows.setdefault(m, []).append('B')

# Candidates per window
candidates_by_window = {
    'A': ['Mysql01','Mysql02','Redis02'],
    'B': ['Mysql02','Mysql01','Redis02']
}

# Helper p95
def p95_series(s):
    s2 = s.dropna()
    return float(np.nanpercentile(s2,95)) if s2.size>0 else np.nan

# ------------------------
# 2) Trace search (broadened)
# ------------------------
minutes_union = list({*window_A_minutes, *window_B_minutes})
trace_subset = trace_df[trace_df['ts_utc8'].isin(minutes_union)].copy()

matched_trace_rows = []
trace_aggs_list = []

if not trace_subset.empty:
    trace_subset['window_labels'] = trace_subset['ts_utc8'].map(lambda t: minute_to_windows.get(t, []))
    trace_expanded = trace_subset.explode('window_labels').rename(columns={'window_labels':'window_label'})
    trace_expanded = trace_expanded[trace_expanded['window_label'].notna()].copy()
    for window_label, cand_list in candidates_by_window.items():
        sub_win = trace_expanded[trace_expanded['window_label']==window_label]
        if sub_win.empty:
            continue
        for cand in cand_list:
            cond1 = sub_win['trace_name'].str.contains(cand, case=False, na=False)
            cond2 = (sub_win['cmdb_id']==cand) & sub_win['trace_name'].str.startswith('trace.self', na=False)
            matched = sub_win[cond1 | cond2].copy()
            if matched.empty:
                continue
            matched['candidate_component'] = cand
            matched['window_label'] = window_label
            matched_trace_rows.append(matched)
            agg = (
                matched.groupby('trace_name', as_index=False)
                .agg(
                    count=('value','count'),
                    mean=('value','mean'),
                    max=('value','max'),
                    p95=('value', p95_series),
                    earliest_ts=('ts_utc8','min')
                )
            )
            agg['window_label'] = window_label
            agg['candidate_component'] = cand
            agg['earliest_timestamp_in_window'] = agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
            trace_aggs_list.append(agg[[
                'window_label','candidate_component','trace_name','count','mean','max','p95','earliest_timestamp_in_window'
            ]])

if matched_trace_rows:
    candidate_traces_broadened = pd.concat(matched_trace_rows, ignore_index=True)
else:
    candidate_traces_broadened = pd.DataFrame(columns=list(trace_df.columns) + ['window_label','candidate_component'])

if trace_aggs_list:
    trace_aggs = pd.concat(trace_aggs_list, ignore_index=True)
    trace_aggs = trace_aggs.sort_values(['window_label','candidate_component','max'], ascending=[True,True,False]).reset_index(drop=True)
    trace_aggs_compact = trace_aggs.head(50).copy()
else:
    trace_aggs_compact = pd.DataFrame(columns=['window_label','candidate_component','trace_name','count','mean','max','p95','earliest_timestamp_in_window'])

# ------------------------
# 3) Error logs & application logs search (broadened)
# ------------------------
# Error logs per window and candidate
err_candidates = ['Mysql01','Mysql02','Redis02']
err_rows = error_df[error_df['ts_utc8'].isin(minutes_union) & error_df['cmdb_id'].isin(err_candidates)].copy()

if not err_rows.empty:
    err_rows['window_labels'] = err_rows['ts_utc8'].map(lambda t: minute_to_windows.get(t, []))
    err_expanded = err_rows.explode('window_labels').rename(columns={'window_labels':'window_label'})
    err_expanded = err_expanded[err_expanded['window_label'].notna()].copy()
else:
    err_expanded = pd.DataFrame(columns=list(error_df.columns)+['window_label'])

rows = []
for window_label, cand_list in candidates_by_window.items():
    for cand in cand_list:
        if cand not in err_candidates:
            continue
        sub = err_expanded[(err_expanded['window_label']==window_label) & (err_expanded['cmdb_id']==cand)]
        if sub.empty:
            rows.append({'window_label':window_label,'cmdb_id':cand,'message_count':0,'sample_messages':None})
        else:
            msgs = sub['message'].astype(str).dropna().unique()[:5].tolist()
            sample_msgs = " || ".join(msgs)
            rows.append({'window_label':window_label,'cmdb_id':cand,'message_count':int(sub.shape[0]),'sample_messages':sample_msgs})

candidate_error_logs_broadened = pd.DataFrame(rows)

# Application logs aggregation
app_components = ['Mysql01','Mysql02','Tomcat01','Tomcat02','Tomcat03','Tomcat04']
log_rows = log_df[log_df['ts_utc8'].isin(minutes_union) & log_df['cmdb_id'].isin(app_components)].copy()

if not log_rows.empty:
    log_rows['window_labels'] = log_rows['ts_utc8'].map(lambda t: minute_to_windows.get(t, []))
    log_expanded = log_rows.explode('window_labels').rename(columns={'window_labels':'window_label'})
    log_expanded = log_expanded[log_expanded['window_label'].notna()].copy()
    df = log_expanded.copy()
    df['val_error'] = np.where(df['log_name']=='log.error_count', df['value'], 0.0)
    df['val_row'] = np.where(df['log_name']=='log.row_count', df['value'], np.nan)
    grp_count = df.groupby(['window_label','cmdb_id','log_name'], as_index=False).agg(count=('value','count'))
    grp_error = df.groupby(['window_label','cmdb_id','log_name'], as_index=False).agg(sum_error_count=('val_error','sum'))
    grp_row = df.groupby(['window_label','cmdb_id','log_name'], as_index=False).agg(max_row_count=('val_row','max'))
    # Merge these on keys
    log_aggs = grp_count.merge(grp_error, on=['window_label','cmdb_id','log_name'], how='left').merge(grp_row, on=['window_label','cmdb_id','log_name'], how='left')
    log_aggs['sum_error_count'] = log_aggs['sum_error_count'].fillna(0.0)
    log_aggs['max_row_count'] = log_aggs['max_row_count'].fillna(0.0)
    # earliest_error_ts
    err_pos = df[(df['log_name']=='log.error_count') & (df['value']>0)].copy()
    if not err_pos.empty:
        earliest_err = err_pos.groupby(['window_label','cmdb_id'], as_index=False).agg(earliest_error_ts=('ts_utc8','min'))
        earliest_err['earliest_error_ts'] = earliest_err['earliest_error_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        earliest_err = earliest_err[['window_label','cmdb_id','earliest_error_ts']]
    else:
        earliest_err = pd.DataFrame(columns=['window_label','cmdb_id','earliest_error_ts'])
    # Merge earliest_error_ts into log_aggs on window_label and cmdb_id
    log_aggs = log_aggs.merge(earliest_err, on=['window_label','cmdb_id'], how='left')
    log_aggs['earliest_error_ts'] = log_aggs['earliest_error_ts'].where(pd.notnull(log_aggs['earliest_error_ts']), None)
    candidate_app_logs_broadened = log_aggs[['window_label','cmdb_id','log_name','count','sum_error_count','max_row_count','earliest_error_ts']].copy()
    candidate_app_logs_broadened = candidate_app_logs_broadened.sort_values(['sum_error_count','max_row_count'], ascending=[False,False]).head(20).reset_index(drop=True)
else:
    candidate_app_logs_broadened = pd.DataFrame(columns=['window_label','cmdb_id','log_name','count','sum_error_count','max_row_count','earliest_error_ts'])

# Store full matched trace rows in kernel variable 'candidate_traces_broadened' (already set)
# Return the compact aggregated tables
trace_aggs_compact, candidate_error_logs_broadened, candidate_app_logs_broadened

```
Out[9]:
```
```python
# Plain-English summary string based on the broadened-search results in the kernel.
# This cell only builds and outputs a concise summary for human reading.

summary = (
    "Broadened trace & log search summary (windows A: 21:58–22:02, B: 22:00–22:04, UTC+8):\n\n"
    "1) Trace findings:\n"
    " - No matching trace rows were found for the candidate components (Mysql01, Mysql02, Redis02) in the ±2-minute windows when searching for trace_name containing the component or trace.self on that component.\n"
    " - In short: no direct per-minute trace evidence tying the metric faults to service-call latency spikes for those candidates at the minute granularity used.\n\n"
    "2) Error log findings:\n"
    " - error_logs checks for Mysql01, Mysql02, Redis02 in both windows returned zero messages (message_count = 0 for all candidates).\n"
    " - Therefore there is no explicit error-log evidence in error_logs.csv at these minutes for those components.\n\n"
    "3) Application log findings (log.csv):\n"
    " - Aggregated app logs in the broadened windows show request-volume metrics (log.row_count) for Tomcat instances and small counts for Mysql, but sum_error_count (log.error_count) is zero across the top aggregates.\n"
    " - No earliest_error_ts was found (no log.error_count > 0) in these aggregates.\n\n"
    "4) Interpretation and diagnosis decision (concise):\n"
    " - Metric-level fault analysis earlier showed heavy and high-severity anomalies concentrated on MySQL-related KPIs (Mysql01 and Mysql02) at the two peak minutes (22:00 and 22:02).\n"
    " - The broadened trace/log search did NOT find corroborating per-minute traces or error-log messages at the minute resolution used.\n"
    " - Combining both pieces: metrics strongly implicate MySQL (Mysql01 primary at 22:00; Mysql02 prominent around 22:02) as the most-downstream faulty service for the two failures. Redis02 is a secondary candidate but with lower severity.\n\n"
    "5) Recommended next steps (practical):\n"
    " - Inspect traces with finer timestamp resolution (include seconds) and expand the time window (±5 minutes) to capture asynchronous events.\n"
    " - Search trace_name patterns case-insensitively and more broadly (substrates, 'to_', 'from_', service IDs) and check full trace traces (not only minute-aligned aggregates).\n"
    " - Inspect raw error log files and application logs across the full incident window (not only exact-minute rows) and check MySQL slow-query logs, lock/contention, connection/queue metrics.\n"
    " - Prioritize MySQL instances (Mysql01 and Mysql02) for immediate investigation: check query latency, locks, connection saturation, and I/O metrics.\n\n"
    "Stored artifacts: candidate_traces_broadened (matched trace rows, empty for the strict rules used),\n"
    "                 candidate_error_logs_broadened (per-candidate error-log summary, zero counts),\n"
    "                 candidate_app_logs_broadened (app-log aggregates with log.row_count present but sum_error_count = 0).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [window_label, candidate_component, trace_name, count, mean, max, p95, earliest_timestamp_in_window]
Index: [],   window_label  cmdb_id  message_count sample_messages
0            A  Mysql01              0            None
1            A  Mysql02              0            None
2            A  Redis02              0            None
3            B  Mysql02              0            None
4            B  Mysql01              0            None
5            B  Redis02              0            None,    window_label   cmdb_id          log_name  count  sum_error_count  max_row_count earliest_error_ts
0             A  Tomcat04     log.row_count      3              0.0          166.0              None
1             B  Tomcat04     log.row_count      5              0.0          166.0              None
2             A  Tomcat02     log.row_count      3              0.0          154.0              None
3             B  Tomcat02     log.row_count      5              0.0          154.0              None
4             A  Tomcat01     log.row_count      3              0.0          152.0              None
5             A  Tomcat03     log.row_count      3              0.0          152.0              None
6             B  Tomcat01     log.row_count      5              0.0          152.0              None
7             B  Tomcat03     log.row_count      5              0.0          152.0              None
8             A   Mysql01     log.row_count      2              0.0            3.0              None
9             B   Mysql01     log.row_count      4              0.0            3.0              None
10            A   Mysql01   log.error_count      2              0.0            0.0              None
11            A  Tomcat01   log.error_count      3              0.0            0.0              None
12            A  Tomcat01  log.gc_pause_max      1              0.0            0.0              None
13            A  Tomcat02   log.error_count      3              0.0            0.0              None
14            A  Tomcat02  log.gc_pause_max      2              0.0            0.0              None
15            A  Tomcat03   log.error_count      3              0.0            0.0              None
16            A  Tomcat03  log.gc_pause_max      1              0.0            0.0              None
17            A  Tomcat04   log.error_count      3              0.0            0.0              None
18            A  Tomcat04  log.gc_pause_max      1              0.0            0.0              None
19            B   Mysql01   log.error_count      4              0.0            0.0              None)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: faults_table, trace_df, error_df
if 'faults_table' not in globals():
    raise RuntimeError("Expected 'faults_table' in kernel.")

# Ensure fault_start_ts is datetime tz-aware in Asia/Shanghai
fts = faults_table.copy()
if 'fault_start_ts' in fts.columns:
    fts['fault_start_ts'] = pd.to_datetime(fts['fault_start_ts'])
    if fts['fault_start_ts'].dt.tz is None:
        fts['fault_start_ts'] = fts['fault_start_ts'].dt.tz_localize(tz)
    else:
        fts['fault_start_ts'] = fts['fault_start_ts'].dt.tz_convert(tz)
else:
    raise RuntimeError("faults_table missing 'fault_start_ts' column.")

# Task 1: per-minute per-component metric ranking for two minutes (epoch seconds)
minutes_epoch = [1615384800, 1615384920]  # 22:00:00 and 22:02:00 UTC+8
minutes_dt = [pd.to_datetime(m, unit='s').tz_localize('UTC').tz_convert(tz) for m in minutes_epoch]
# Normalize to minute (should already be minute-aligned)
minutes_dt = [m.floor('min') for m in minutes_dt]

rows = []
for m in minutes_dt:
    sub = fts[fts['fault_start_ts'] == m]
    if sub.empty:
        continue
    agg = (
        sub.groupby('cmdb_id', as_index=False)
        .agg(
            faults_count=('cmdb_id','size'),
            distinct_kpis_with_faults=('kpi_name', lambda s: s.nunique()),
            max_severity=('severity','max'),
            earliest_fault_start_ts=('fault_start_ts','min')
        )
    )
    agg['minute_ts'] = m
    rows.append(agg)

if rows:
    minute_component_rankings = pd.concat(rows, ignore_index=True)
    # Format earliest timestamp for compact display
    minute_component_rankings['earliest_fault_start_ts'] = minute_component_rankings['earliest_fault_start_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Order and limit top 10 per minute by max_severity desc then faults_count desc
    minute_component_rankings = minute_component_rankings.sort_values(['minute_ts','max_severity','faults_count'], ascending=[True,False,False])
    minute_component_rankings = minute_component_rankings.groupby('minute_ts').head(10).reset_index(drop=True)
    # Make minute_ts string column for display
    minute_component_rankings['minute_ts'] = minute_component_rankings['minute_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Select columns as requested
    minute_component_rankings = minute_component_rankings[['minute_ts','cmdb_id','faults_count','distinct_kpis_with_faults','max_severity','earliest_fault_start_ts']]
else:
    minute_component_rankings = pd.DataFrame(columns=['minute_ts','cmdb_id','faults_count','distinct_kpis_with_faults','max_severity','earliest_fault_start_ts'])

# Store as variable name required
minute_component_rankings = minute_component_rankings

# Task 2: broader name-based trace and error-log correlation for top candidates
# Take up to top 3 components per minute from Task 1 (by ordering already)
candidates = []
for m in minute_component_rankings['minute_ts'].drop_duplicates():
    top3 = minute_component_rankings[minute_component_rankings['minute_ts']==m].head(3)['cmdb_id'].tolist()
    candidates.extend(top3)
# Deduplicate while preserving order
seen = set()
candidates_unique = []
for c in candidates:
    if c not in seen:
        seen.add(c)
        candidates_unique.append(c)

# Build candidate substrings: lowercase service type tokens plus exact cmdb_id
candidate_substrings = set()
for c in candidates_unique:
    candidate_substrings.add(c.lower())  # e.g., 'mysql01'
    # also add generic short names 'mysql' or 'redis' if present
    if 'mysql' in c.lower():
        candidate_substrings.add('mysql')
    if 'redis' in c.lower():
        candidate_substrings.add('redis')

candidate_substrings = list(candidate_substrings)

# Incident full window: 1615384800 — 1615386600 (22:00 — 22:30 UTC+8)
full_start = pd.to_datetime(1615384800, unit='s').tz_localize('UTC').tz_convert(tz)
full_end = pd.to_datetime(1615386600, unit='s').tz_localize('UTC').tz_convert(tz)

# Prepare trace_df timezone and filter to window
trace_all = trace_df.copy()
if 'ts_utc8' not in trace_all.columns:
    trace_all['timestamp'] = pd.to_numeric(trace_all['timestamp'], errors='coerce').astype('Int64')
    trace_all['ts_utc8'] = pd.to_datetime(trace_all['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

trace_window = trace_all[(trace_all['ts_utc8'] >= full_start) & (trace_all['ts_utc8'] <= full_end)].copy()

# Build matching condition: trace_name contains any candidate substring (case-insensitive)
if not trace_window.empty and candidates_unique:
    # Lowercase trace_name for matching
    trace_window['trace_name_lc'] = trace_window['trace_name'].astype(str).str.lower()
    mask = pd.Series(False, index=trace_window.index)
    for sub in candidate_substrings:
        mask = mask | trace_window['trace_name_lc'].str.contains(sub, na=False)
    # Also OR condition: trace_name contains exact cmdb_id token (case-insensitive) or (cmdb_id == candidate and trace_name startswith 'trace.self')
    for c in candidates_unique:
        mask = mask | trace_window['trace_name_lc'].str.contains(c.lower(), na=False)
        mask = mask | ((trace_window['cmdb_id']==c) & trace_window['trace_name'].str.startswith('trace.self', na=False))
    matched_trace = trace_window[mask].copy()
else:
    matched_trace = trace_window.iloc[0:0].copy()

# Aggregate matched_trace by trace_name
if not matched_trace.empty:
    def p95_series(s):
        s2 = s.dropna()
        return float(np.nanpercentile(s2,95)) if s2.size>0 else np.nan
    trace_candidates_by_name = (
        matched_trace
        .groupby('trace_name', as_index=False)
        .agg(
            count=('value','count'),
            mean=('value','mean'),
            max=('value','max'),
            p95=('value', p95_series),
            earliest_ts=('ts_utc8','min')
        )
    )
    trace_candidates_by_name['earliest_timestamp_in_window'] = trace_candidates_by_name['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    trace_candidates_by_name = trace_candidates_by_name[['trace_name','count','mean','max','p95','earliest_timestamp_in_window']].sort_values('max', ascending=False).head(20).reset_index(drop=True)
else:
    trace_candidates_by_name = pd.DataFrame(columns=['trace_name','count','mean','max','p95','earliest_timestamp_in_window'])

# Store the matched_trace full rows as variable for follow-up
trace_candidates_by_name = trace_candidates_by_name
trace_candidates_full_rows = matched_trace  # stored but not displayed

# Error logs across full window for candidate cmdb_ids
err_all = error_df.copy()
if 'ts_utc8' not in err_all.columns:
    err_all['timestamp'] = pd.to_numeric(err_all['timestamp'], errors='coerce').astype('Int64')
    err_all['ts_utc8'] = pd.to_datetime(err_all['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

err_window = err_all[(err_all['ts_utc8'] >= full_start) & (err_all['ts_utc8'] <= full_end) & (err_all['cmdb_id'].isin(candidates_unique))].copy()

rows = []
for c in candidates_unique:
    sub = err_window[err_window['cmdb_id']==c]
    if sub.empty:
        rows.append({'cmdb_id':c,'message_count':0,'sample_messages':None})
    else:
        msgs = sub['message'].astype(str).dropna().unique()[:5].tolist()
        sample_msgs = " || ".join(msgs) if len(msgs)>0 else None
        rows.append({'cmdb_id':c,'message_count':int(sub.shape[0]),'sample_messages':sample_msgs})

error_logs_candidates = pd.DataFrame(rows)

# Store results in kernel variables as required
minute_component_rankings = minute_component_rankings
trace_candidates_by_name = trace_candidates_by_name
error_logs_candidates = error_logs_candidates

# Return only the three compact aggregate tables
minute_component_rankings, trace_candidates_by_name, error_logs_candidates

```
Out[10]:
```
Summary (plain English):

- Per-minute metric ranking:
  - 2021-03-10 22:00 (UTC+8): Mysql01 is the top candidate (137 faults starting, max_severity ≈ 19.0), followed by Mysql02 and Redis02.
  - 2021-03-10 22:02 (UTC+8): Mysql02 and Mysql01 are prominent (Mysql02 shows higher faults_count at this minute; IG02 and Tomcat03 also show elevated severity but much smaller than MySQL peaks).

- Trace correlation (name-based search across the incident window):
  - No strong trace entries directly containing "mysql" or "redis" matched with high peak values.
  - The dominant trace aggregates found are trace.self.row_count (high counts) and traces involving IG02 and Tomcat03 (row_count and duration p95 peaks). Those indicate heavy request/traffic through IG/Tomcat, but not a direct per-minute trace signature tying the metric anomalies to MySQL RPCs at the minute granularity used.

- Error-log check:
  - error_logs.csv returned no matching error messages for Mysql01, Mysql02, or Redis02 in the inspected windows (message_count = 0).

- Conclusion / most-downstream faulty service for each failure:
  - 22:00 failure: most likely Mysql01 (highest fault count and highest severity).
  - 22:02 failure: most likely Mysql02 (prominent faults_count and elevated severity).
  - Redis02 is a secondary candidate but with lower severity than MySQL at these minutes.

- Caveats & recommended next steps:
  - There is little direct trace or error-log corroboration at strict minute boundaries. Broaden trace/time matching (include seconds, ±5 min), search trace_name patterns more flexibly, and inspect raw MySQL logs (slow queries, locks, connection limits) and full error log files over the incident window.
  - Prioritize MySQL instances (Mysql01, Mysql02) for immediate investigation (I/O, locks, connections, slow queries).

The original code execution output of IPython Kernel is also provided below for reference:

(                  minute_ts   cmdb_id  faults_count  distinct_kpis_with_faults  max_severity  earliest_fault_start_ts
0   2021-03-10 22:00:00 CST   Mysql01           137                        137     19.000000  2021-03-10 22:00:00 CST
1   2021-03-10 22:00:00 CST   Mysql02           124                        124      3.000000  2021-03-10 22:00:00 CST
2   2021-03-10 22:00:00 CST   Redis02            54                         54      3.000000  2021-03-10 22:00:00 CST
3   2021-03-10 22:00:00 CST  apache02            12                         12      0.333333  2021-03-10 22:00:00 CST
4   2021-03-10 22:00:00 CST  apache01            47                         47      0.200018  2021-03-10 22:00:00 CST
5   2021-03-10 22:00:00 CST      IG02            37                         37      0.142857  2021-03-10 22:00:00 CST
6   2021-03-10 22:00:00 CST  Tomcat02            27                         27      0.071578  2021-03-10 22:00:00 CST
7   2021-03-10 22:00:00 CST   Redis01            37                         37      0.061224  2021-03-10 22:00:00 CST
8   2021-03-10 22:00:00 CST      MG01            15                         15      0.040165  2021-03-10 22:00:00 CST
9   2021-03-10 22:00:00 CST      IG01            35                         35      0.014458  2021-03-10 22:00:00 CST
10  2021-03-10 22:02:00 CST      IG02            40                         40      0.339754  2021-03-10 22:02:00 CST
11  2021-03-10 22:02:00 CST  Tomcat03            39                         39      0.310345  2021-03-10 22:02:00 CST
12  2021-03-10 22:02:00 CST   Redis02            47                         47      0.206387  2021-03-10 22:02:00 CST
13  2021-03-10 22:02:00 CST   Mysql02           140                        140      0.200704  2021-03-10 22:02:00 CST
14  2021-03-10 22:02:00 CST   Mysql01           100                        100      0.200000  2021-03-10 22:02:00 CST
15  2021-03-10 22:02:00 CST  apache01            29                         29      0.157149  2021-03-10 22:02:00 CST
16  2021-03-10 22:02:00 CST      IG01            39                         39      0.127243  2021-03-10 22:02:00 CST
17  2021-03-10 22:02:00 CST      MG02            15                         15      0.100389  2021-03-10 22:02:00 CST
18  2021-03-10 22:02:00 CST  apache02            10                         10      0.066830  2021-03-10 22:02:00 CST
19  2021-03-10 22:02:00 CST  Tomcat01             3                          3      0.017341  2021-03-10 22:02:00 CST,                            trace_name  count        mean          max          p95 earliest_timestamp_in_window
0                trace.self.row_count     60  673.183333  1068.000000  1008.250000      2021-03-10 22:00:00 CST
1           trace.from_IG02.row_count    120   62.000000    89.000000    80.000000      2021-03-10 22:00:00 CST
2         trace.to_Tomcat03.row_count     60   62.583333    89.000000    79.050000      2021-03-10 22:00:00 CST
3       trace.from_Tomcat03.row_count     60   62.633333    86.000000    82.050000      2021-03-10 22:00:00 CST
4    trace.from_Tomcat03.duration_p95     60    0.591924     1.434400     1.079480      2021-03-10 22:00:00 CST
5      trace.to_Tomcat03.duration_p95     60    0.669207     1.231600     1.102462      2021-03-10 22:00:00 CST
6        trace.from_IG02.duration_p95    120    0.732366     1.166800     1.092402      2021-03-10 22:00:00 CST
7             trace.self.duration_p95     60    0.440019     1.106600     1.063098      2021-03-10 22:00:00 CST
8       trace.from_IG02.duration_mean    120    0.127804     0.251197     0.222722      2021-03-10 22:00:00 CST
9     trace.to_Tomcat03.duration_mean     60    0.122565     0.236893     0.220441      2021-03-10 22:00:00 CST
10  trace.from_Tomcat03.duration_mean     60    0.100709     0.220133     0.171013      2021-03-10 22:00:00 CST
11           trace.self.duration_mean     60    0.073419     0.214244     0.172112      2021-03-10 22:00:00 CST,     cmdb_id  message_count sample_messages
0   Mysql01              0            None
1   Mysql02              0            None
2   Redis02              0            None
3      IG02              0            None
4  Tomcat03              0            None)```
```